# SparkX-Loop v0.2 — Colab G4 (96 GB VRAM / 130 GB RAM, 23-Hour) Full-Finetuning & Memory-Tiered MoE-Engram

This one-shot notebook initializes the [`BaiTian6641/Spark-X2.6-Recursion`](https://github.com/BaiTian6641/Spark-X2.6-Recursion) base environment on a **Google Colab G4 instance (96 GB VRAM + 130 GB System RAM)** and modifies **`XHToken/Spark-X2.6-4B`** into a **60-effective-layer Looped Recursive Transformer** with:

1. **Sublayer Dual-mHC (`mhc_attn` + `mhc_ffn`) & Pass-Indexed Deliberation Adapters**:
   - Wraps both attention and FFN sublayers in recurrent layers `[4, 28)` with token-dependent FP32 RMSNorm + Sinkhorn Birkhoff routing (`96` Sinkhorn steps across 2 passes) and zero-initialized SwiGLU deliberation adapters (`rank=16`, `max_passes=4`).
2. **Memory-Tiered MoE-Engram Backbone Architecture (`TieredMoEEngram` / `HashedBigramEngram`)**:
   - Inserted after the Prelude (`Layer 3`) and at every **4-layer Spark macro-block anchor (`Layers 7, 11, 15, 19, 23, 27`)** inside the recurrent core (`7` Engram layers × `16` experts = `112` MoE memory experts across the backbone).
   - **Tier 1 (Lexical N-Gram Tier)**: $O(1)$ multi-head causal modular hash lookup over `2-gram`, `3-gram`, and `skip-2-gram` token windows (zero matmul FLOPs, reused across recurrent passes).
   - **Tier 2 (Hot Shared Knowledge Tier)**: DeepSeekMoE-style always-active compact shared expert (`shared_down -> SiLU -> shared_up`) for universal reasoning primitives.
   - **Tier 3 (Cold/Domain Sparse Routed Memory Experts)**: MoE + PEER/UltraMemV2 Top-$K$ (`K=2` of `16`) memory experts with pass-conditioned routing (`pass_router_bias`) and auxiliary-loss-free load balancing (`expert_bias`, `expert_load_ema`).
   - **Gentle Engram Formation**: Zero-initialized output projection (`project.weight = 0`) preserves exact 100% `Spark-X2.6-4B` behavior at Step 0 while compacting new knowledge during full finetuning.
3. **1M Token Context Window Preservation (`patch_spark_attention_for_long_context`)**:
   - Replaces upstream `Spark2_5Attention`'s $O(T^2)$ `eager_attention_forward` with fused PyTorch `F.scaled_dot_product_attention(..., enable_gqa=True)`, preserving `3 × sliding_attention (w=512, RoPE) + 1 × full_attention (NoPE)` up to 1M tokens.
4. **Curated Multi-Domain Reasoning, Scientific & Multi-Turn Conversation Mixture**:
   - **Claude Fable & Claude Opus (4.6/4.7/4.8)** CoT traces
   - **GPT Astra / GPT-6 / GPT-5** step-by-step reasoning
   - **Scientific Reasoning**: `nvidia/OpenScienceReasoning-2`, `Jackrong/Kimi-K2.5-Reasoning-1M-Cleaned` (`PHD-Science`), `MegaScience/MegaScience`, `MegaScience/TextbookReasoning`, `OpenDataArena/Spark-234K`, `HuggingFaceTB/smoltalk2` (`OpenThoughts3_1.2M`)
   - **Multi-Turn Conversation**: `nvidia/Nemotron-Instruction-Following-Chat-v1`, `NousResearch/Hermes-3-Dataset`, `HuggingFaceH4/ultrachat_200k`, `DataCreatorAI/Multi-Turn-Conversational-SFT`
   - **OpenCode Supervised CoT Distillation** (`OPENCODE_API_KEY`: `deepseek-v4-flash`, `haiku5.5`, `mimo-v2.6-pro`) + **GSM8K** verification.


In [ ]:
import os, sys, subprocess, time, shutil, pathlib

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = pathlib.Path('/content/Spark-X2.6-Recursion')
if pathlib.Path('/content').exists():
    if not REPO_DIR.exists():
        print('Cloning base repo:', REPO_URL)
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=False)
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
        subprocess.run(['git', 'pull', '--ff-only'], check=False)
print('Working directory:', os.getcwd())

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate>=1.10,<2', 'datasets>=3,<5',
    'huggingface_hub>=0.34,<2', 'safetensors>=0.5', 'openai>=1.5',
    'bitsandbytes>=0.45.0', 'cut-cross-entropy'])
# Install Unsloth / Unsloth Studio kernels without overriding Colab's native Blackwell PyTorch or transformers==4.57.1
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'unsloth', 'unsloth_zoo'], check=False)
try:
    import unsloth_zoo
    UNSLOTH_AVAILABLE = True
except Exception:
    UNSLOTH_AVAILABLE = False
print('Dependencies installed | Unsloth Studio utilities available:', UNSLOTH_AVAILABLE)

In [ ]:
# ===== USER SETTINGS: tuned for Colab G4 (96 GB VRAM + 130 GB System RAM, 23-Hour Run) =====
CFG = {
 'MODEL_ID': 'XHToken/Spark-X2.6-4B',
 'MODEL_REVISION': 'main',   # verified commit: d10f96e0abf4cb7ca0a06a442bc5cf55645fd74d
 'TRAIN_MODE': 'auto',       # 'auto' -> 'full' (all 36 layers + Dual-mHC + Backbone MoE-Engrams + LoRA) on >=80 GiB G4
 'MAX_SEQ_LENGTH': 'auto',   # 'auto' -> 4096 on >=85 GiB (Colab G4), 2048 on >=65 GiB, 1024 on >=45 GiB, else 512
 'PACK_SEQUENCES': True,     # pack shorter CoT sequences in 130 GB RAM up to MAX_SEQ_LENGTH for peak G4 utilization
 'TRAIN_EXAMPLES': 28000,    # streamed CoT, Science & Multi-Turn Chat examples in 130 GB System RAM
 'EVAL_EXAMPLES': 96,
 'GRAD_ACCUM_STEPS': 8,
 'LR_ADAPTER': 1.5e-4,       # learning rate for Memory-Tiered MoE-Engrams, Pass Adapters & LoRA
 'LR_MHC': 5.0e-3,           # dedicated routing group (0 weight decay) for Dual-mHC & MoE-Engram gates
 'LR_FULL': 1.5e-5,          # gentle backbone learning rate for all 36 decoder layers under PagedAdamW8bit
 'USE_8BIT_OPTIM': True,     # Unsloth / bitsandbytes PagedAdamW8bit (pages to 130 GB System RAM if needed)
 'LORA_RANK': 'auto',        # 'auto' -> 16 on >=80 GiB (Colab G4), else 8
 'LORA_ALPHA': 'auto',       # 'auto' -> 32 on >=80 GiB (Colab G4), else 16
 'DYNAMIC_MHC': True,        # SenseNova-Loop style token-dependent RMSNorm + FP32 linear Sinkhorn router
 'SUBLAYER_MHC': True,       # Dual-mHC: independent mhc_attn + mhc_ffn per recurrent decoder layer
 'PASS_ADAPTER_RANK': 16,    # Pass-conditioned low-rank deliberation adapter inside each recurrent layer
 'CE_CHUNK_TOKENS': 512,     # Token-chunked masked cross-entropy (avoids [B, T, 167168] FP32 logits spike)
 'OFFLOAD_CKPT_CPU': 'auto', # 'auto' -> True if MAX_SEQ_LENGTH >= 8192 (uses 130 GB System RAM pinned memory)
 'ENGRAM': True,
 'ENG_BUCKETS': 16384,
 'ENG_DIM': 64,
 'ENG_EXPERTS': 16,          # Tier-3 MoE routed memory experts per Engram layer (7 layers x 16 = 112 experts)
 'ENG_TOP_K': 2,             # Sparse Top-2 active memory experts per token
 'ENG_EXPERT_RANK': 32,      # PEER / UltraMemV2 inner value slots per expert
 'ENG_SHARED_RANK': 32,      # Tier-2 Hot Shared Expert bottleneck rank
 'BACKBONE_ENGRAM_STRIDE': 4,# Insert Backbone MoE-Engrams at every 4-layer Spark macro-block anchor [7,11,15,19,23,27]
 'FULL_LAYERS': 'auto',      # used when TRAIN_MODE='hybrid' ([32,33,34,35] on >=80 GiB); 'full' unfreezes [0..35]
 'MAX_RUN_HOURS': 22.5,      # 22.5h active training budget inside a 23h G4 run (leaves 30m for final HF export)
 'SAVE_EVERY_STEPS': 250,    # periodic intra-stage checkpoint export to disk / HF Hub
 'STAGES': [
    {'name':'warmup', 'start':4, 'end':12, 'passes':1, 'steps':150},
    {'name':'R8',     'start':4, 'end':12, 'passes':2, 'steps':350},
    {'name':'R12',    'start':4, 'end':16, 'passes':2, 'steps':600},
    {'name':'R24',    'start':4, 'end':28, 'passes':2, 'steps':2500},
 ],
 'REASONING_DATASETS': [
    # 1. Claude Fable Series (CoT & Chat Distillation)
    {'id': 'kelexine/fable-5-sft-traces', 'split': 'train', 'family': 'claude-fable', 'max_rows': 2500},
    {'id': 'HelioAI/Claude-Fable-5-5500x', 'split': 'train', 'family': 'claude-fable', 'max_rows': 2500},
    {'id': 'MoreThought/Fable-5.1-Max-Reasoning-Filtered-10000x', 'split': 'train', 'family': 'claude-fable', 'max_rows': 2000},
    {'id': 'saidutta69/fable-5-premium-v2', 'data_files': 'openai_chat/train.parquet', 'split': 'train', 'family': 'claude-fable', 'max_rows': 1500},
    {'id': 'armand0e/Fable-5-Chat', 'split': 'train', 'family': 'claude-fable', 'max_rows': 350},
    # 2. Claude Opus Series (Opus 4.6 / 4.7 / 4.8 Extended Thinking CoT)
    {'id': 'Roman1111111/claude-opus-4.6-10000x', 'split': 'train', 'family': 'claude-opus', 'max_rows': 2500},
    {'id': 'TeichAI/lordx64-claude-opus-4.7-max-cleaned', 'split': 'train', 'family': 'claude-opus', 'max_rows': 2500},
    {'id': 'angrygiraffe/claude-opus-4.6-4.7-reasoning-8.7k', 'split': 'train', 'family': 'claude-opus', 'max_rows': 2000},
    {'id': 'ronaldcmz/claude_opus_4.8_max_thinking_5k_v2', 'split': 'train', 'family': 'claude-opus', 'max_rows': 1500},
    {'id': 'TeichAI/Claude-Opus-4.6-Reasoning-887x', 'split': 'train', 'family': 'claude-opus', 'max_rows': 880},
    # 3. GPT Astra / GPT-6 / GPT-5 Series (Scientific & High-Density Step-by-Step CoT)
    {'id': 'FlaneAI/GPT-6Astra-scientific-research', 'split': 'train', 'family': 'gpt-astra-6', 'max_rows': 100},
    {'id': 'Roman1111111/gpt-5.4-step-by-step-reasoning', 'split': 'train', 'family': 'gpt-5-reasoning', 'max_rows': 1500},
    {'id': 'vanty120/Gpt-5.4-Xhigh-Reasoning-2000x', 'split': 'train', 'family': 'gpt-5-reasoning', 'max_rows': 1500},
    {'id': 'TeichAI/gpt-5.1-high-reasoning-1000x', 'split': 'train', 'family': 'gpt-5-reasoning', 'max_rows': 1000},
    {
        'id': 'Manusagents/GPT-5.5-Gemini-3.1-Pro-Grok-4-Claude-Fable-5-Mythos-5-Qwen-3.7-Max-and-more-Distillation-Dataset',
        'data_files': 'data/distilled/gpt55_thinking/shard-00000.jsonl',
        'split': 'train',
        'family': 'gpt-5.5-thinking',
        'max_rows': 1500,
    },
    # 4. Scientific Reasoning & PhD/University STEM CoT Series
    {'id': 'nvidia/OpenScienceReasoning-2', 'split': 'train', 'family': 'science-reasoning', 'max_rows': 2500},
    {'id': 'Jackrong/Kimi-K2.5-Reasoning-1M-Cleaned', 'config': 'PHD-Science', 'split': 'train', 'family': 'science-reasoning', 'max_rows': 2000},
    {'id': 'MegaScience/MegaScience', 'split': 'train', 'family': 'science-reasoning', 'max_rows': 2000},
    {'id': 'MegaScience/TextbookReasoning', 'split': 'train', 'family': 'science-reasoning', 'max_rows': 1500},
    {'id': 'OpenDataArena/Spark-234K', 'split': 'train', 'family': 'science-reasoning', 'max_rows': 1500},
    {'id': 'HuggingFaceTB/smoltalk2', 'config': 'Mid', 'split': 'OpenThoughts3_1.2M', 'family': 'science-reasoning', 'max_rows': 1500},
    # 5. High-Quality Multi-Turn Conversation & Instruction-Following Series
    {'id': 'nvidia/Nemotron-Instruction-Following-Chat-v1', 'split': 'chat_if', 'family': 'multi-turn-chat', 'max_rows': 2000},
    {'id': 'NousResearch/Hermes-3-Dataset', 'split': 'train', 'family': 'multi-turn-chat', 'max_rows': 1500},
    {'id': 'HuggingFaceH4/ultrachat_200k', 'split': 'train_sft', 'family': 'multi-turn-chat', 'max_rows': 1500},
    {'id': 'DataCreatorAI/Multi-Turn-Conversational-SFT', 'split': 'train', 'family': 'multi-turn-chat', 'max_rows': 1000},
    # 6. Math Verification Anchor (GSM8K step-by-step CoT split into <think>)
    {'id': 'openai/gsm8k', 'config': 'main', 'split': 'train', 'family': 'gsm8k-cot', 'max_rows': 1500},
 ],
 'EVAL_LIMIT_PER_STAGE': 32,
 'PUSH_TO_HF': True,         # uploads stages & full/trainable weights to https://huggingface.co/TassP0lar
 'HF_REPO_ID': 'TassP0lar/Spark-X2.6-4B-Loop-mHC',
 'HF_PRIVATE': True,
 'UPLOAD_EACH_STAGE': True,
 'ENABLE_TEACHER': 'auto',   # 'auto' -> True if OPENCODE_API_KEY is present in Colab Secrets
 'TEACHER_EXAMPLES': 48,
 'TEACHER_MODELS': ['deepseek-v4-flash', 'haiku5.5', 'mimo-v2.6-pro'],
 'OPENCODE_BASE_URL': 'https://opencode.ai/inference/openai/v1',
 'TEACHER_MAX_TOKENS': 1024,
}


In [ ]:
import json, random, math, re, gc, copy, pathlib
from concurrent.futures import ThreadPoolExecutor
from decimal import Decimal, InvalidOperation
import torch
from google.colab import userdata
from huggingface_hub import HfApi

STARTED = time.monotonic()
random.seed(2026); torch.manual_seed(2026)
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for this 4B experiment. Runtime > Change runtime type > GPU (G4 preferred).')
props = torch.cuda.get_device_properties(0)
GPU_GIB = props.total_memory / (2**30)
SYS_RAM_GIB = (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')) / (2**30) if hasattr(os, 'sysconf') else 0.0
print(f'GPU: {props.name} | {GPU_GIB:.1f} GiB VRAM | {SYS_RAM_GIB:.1f} GiB System RAM | CUDA {torch.version.cuda}')
if GPU_GIB < 20:
    raise RuntimeError('Less than 20 GiB VRAM. Select a 24+ GB GPU; Colab G4 (~96 GB VRAM / ~130 GB RAM) preferred.')
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
if DTYPE != torch.bfloat16:
    print('WARNING: BF16 not supported on this GPU; FP16 path is less stable and unqualified.')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def secret(*names):
    for name in names:
        value = os.environ.get(name)
        if value: return value
        try:
            value = userdata.get(name)
            if value: return value
        except Exception: pass
    return None

HF_TOKEN = secret('HF_TOKEN', 'HF_TOKEM')
OPENCODE_API_KEY = secret('OPENCODE_API_KEY', 'OPENCODE_API')
if CFG['TRAIN_MODE'] == 'auto':
    CFG['TRAIN_MODE'] = 'full' if GPU_GIB >= 80 else 'hybrid'
if CFG['MAX_SEQ_LENGTH'] == 'auto':
    CFG['MAX_SEQ_LENGTH'] = 4096 if GPU_GIB >= 85 else (2048 if GPU_GIB >= 65 else (1024 if GPU_GIB >= 45 else 512))
if CFG['LORA_RANK'] == 'auto':
    CFG['LORA_RANK'] = 16 if GPU_GIB >= 80 else 8
if CFG['LORA_ALPHA'] == 'auto':
    CFG['LORA_ALPHA'] = 32 if GPU_GIB >= 80 else 16
if CFG['FULL_LAYERS'] == 'auto':
    CFG['FULL_LAYERS'] = list(range(36)) if CFG['TRAIN_MODE'] == 'full' else ([32, 33, 34, 35] if GPU_GIB >= 80 else ([32, 33] if GPU_GIB >= 55 else []))
if CFG['OFFLOAD_CKPT_CPU'] == 'auto':
    CFG['OFFLOAD_CKPT_CPU'] = bool(CFG['MAX_SEQ_LENGTH'] >= 8192 and SYS_RAM_GIB >= 64)
if CFG['ENABLE_TEACHER'] == 'auto':
    CFG['ENABLE_TEACHER'] = bool(OPENCODE_API_KEY and CFG.get('TEACHER_MODELS'))
if CFG['ENABLE_TEACHER'] and not OPENCODE_API_KEY:
    print('WARNING: ENABLE_TEACHER was True but OPENCODE_API_KEY is missing; disabling teacher pass.')
    CFG['ENABLE_TEACHER'] = False
if CFG['PUSH_TO_HF'] and not HF_TOKEN:
    raise RuntimeError('Set HF_TOKEN in Colab Secrets before the one-shot run (target: https://huggingface.co/TassP0lar).')
assert CFG['MAX_RUN_HOURS'] <= 23.5, 'Leave at least 30 minutes for final export before the 24-hour Colab hard cap.'
print('Resolved Hardware Config:', {k:v for k,v in CFG.items() if k not in ('REASONING_DATASETS', 'FULL_LAYERS')})
print(f'Train mode: {CFG["TRAIN_MODE"]} | Unfrozen physical layers: {len(CFG["FULL_LAYERS"])}/36 | OpenCode teacher enabled: {CFG["ENABLE_TEACHER"]}')

## Core implementation

This cell writes the full Python implementation into the *current* Colab session. It is embedded in the notebook, so no separate file download is necessary. The same source is copied into the exported model repo for subsequent loading.

In [ ]:
%%writefile sparkx_loop_core.py
"""SparkX-Loop v0.2: Looped Recursive Transformer with Sublayer Dual-mHC, Pass-Conditioned
Deliberation Adapters, Multi-Scale Causal Engram, 1M-Context Fused SDPA, and Unsloth/Chunked CE.

Intended upstream: XHToken/Spark-X2.6-4B, transformers==4.57.1.
- Preserves Spark-X2.6-4B's 1M context window architecture (3 x sliding_attention w=512 RoPE +
  1 x full_attention NoPE per 4-layer macro-block, plus headwise sigmoid output gating g_proj).
- Upgrades recurrent layers [4, 28) with SenseNova-Loop-inspired sublayer Dual-mHC (mhc_attn + mhc_ffn),
  token-dependent FP32 RMSNorm Sinkhorn routing, pass-indexed conditioning, and zero-gated identity init.
- Supports both full-parameter finetuning (all 36 physical layers + mHC + Engram on 96 GB G4) and
  hybrid LoRA + selective layer finetuning.
- Supports fast per-visit KV-cached looped generation (`generate_cached` via `_VisitCache`) and
  exact full-recompute generation (`generate_slow`).
"""
from __future__ import annotations
from contextlib import nullcontext
import copy
import dataclasses
import importlib
import inspect
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

_SDPA_SUPPORTS_GQA = (
    hasattr(F, 'scaled_dot_product_attention')
    and 'enable_gqa' in inspect.signature(F.scaled_dot_product_attention).parameters
)


@dataclasses.dataclass
class LoopSettings:
    start: int = 4
    end: int = 28           # exclusive: 6 x 4-layer Spark macro-blocks (18 sliding + 6 full NoPE)
    passes: int = 2
    max_passes: int = 4     # supports test-time scaling up to 4 recurrent passes
    mhc_streams: int = 2
    eng_buckets: int = 16384
    eng_dim: int = 64
    eng_experts: int = 8            # Tier-3 MoE routed memory experts per Engram layer
    eng_top_k: int = 2              # Sparse Top-K active memory experts per token
    eng_expert_rank: int = 16       # PEER / UltraMemV2 inner value slots per expert
    eng_shared_rank: int = 16       # Tier-2 Hot Shared Expert bottleneck rank
    backbone_engram_stride: int = 4 # Insert Backbone MoE-Engram at each 4-layer Spark macro-block anchor
    checkpoint_layers: bool = True
    offload_checkpoint_to_cpu: bool = False  # uses 130 GB System RAM pinned memory when True
    use_mhc: bool = True
    use_engram: bool = True
    dynamic_mhc: bool = True
    sublayer_mhc: bool = True   # Dual-mHC: independent mhc_attn + mhc_ffn per recurrent layer
    pass_adapter_rank: int = 16 # Pass-indexed low-rank deliberation adapter inside mHC
    ce_chunk_tokens: int = 512  # Token-chunked masked CE to avoid [B, T, 167168] FP32 logits spike


class _CacheLayersView:
    """Expose the virtual execution cache slot when a physical layer indexes cache.layers."""
    def __init__(self, cache, physical: int, virtual: int):
        self.cache = cache
        self.physical = physical
        self.virtual = virtual

    def __getitem__(self, index: int):
        if index != self.physical:
            raise IndexError('A physical layer may only access its own virtual cache slot')
        return self.cache.layers[self.virtual]


class _VisitCache:
    """Redirects physical layer_idx KV cache updates to unique per-visit virtual_idx slots.

    Mirrors the SenseNova-Loop-9B-Preview _VisitCache design so each recurrent pass
    over physical layers [loop_start, loop_end) maintains an independent KV cache slot.
    """
    def __init__(self, cache, physical_idx: int, virtual_idx: int):
        self.cache = cache
        self.physical_idx = physical_idx
        self.virtual_idx = virtual_idx
        if hasattr(cache, 'layers'):
            self.layers = _CacheLayersView(cache, physical_idx, virtual_idx)

    def update(self, key_states, value_states, layer_idx: int, cache_kwargs=None):
        target = self.virtual_idx if layer_idx == self.physical_idx else layer_idx
        return self.cache.update(key_states, value_states, target, cache_kwargs)

    def __getattr__(self, name):
        return getattr(self.cache, name)


def patch_spark_attention_for_long_context(base_model: nn.Module) -> int:
    """Replace Spark2_5Attention's O(T^2) eager_attention_forward with fused PyTorch SDPA / GQA.

    Preserves Spark-X2.6-4B's 1M context window architecture (sliding_window=512 RoPE on
    local layers + NoPE on full_attention layers + headwise sigmoid g_proj gate) while
    avoiding materializing the [B, 32, T, T] attention matrix and 4x repeat_kv expansion.
    """
    if not hasattr(base_model, 'model') or not hasattr(base_model.model, 'layers'):
        return 0
    layers = base_model.model.layers
    if not layers or not hasattr(layers[0], 'self_attn'):
        return 0
    attn_mod = importlib.import_module(type(layers[0].self_attn).__module__)
    if not hasattr(attn_mod, 'eager_attention_forward'):
        return 0
    if getattr(attn_mod, '_sparkx_sdpa_patched', False):
        return len(layers)

    orig_repeat_kv = getattr(attn_mod, 'repeat_kv', None)

    def fused_sdpa_attention_forward(
        module: nn.Module,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        scaling: Optional[float] = None,
        dropout: float = 0.0,
        **kwargs,
    ):
        if scaling is None:
            scaling = 1.0 / math.sqrt(query.shape[-1])
        attn_mask = None
        if attention_mask is not None:
            attn_mask = attention_mask[:, :, :, : key.shape[-2]]
        use_gqa = _SDPA_SUPPORTS_GQA and (module.num_key_value_groups > 1)
        if not use_gqa and orig_repeat_kv is not None and module.num_key_value_groups > 1:
            key = orig_repeat_kv(key, module.num_key_value_groups)
            value = orig_repeat_kv(value, module.num_key_value_groups)
        sdpa_kwargs = {'scale': scaling}
        if use_gqa:
            sdpa_kwargs['enable_gqa'] = True
        out = F.scaled_dot_product_attention(
            query,
            key,
            value,
            attn_mask=attn_mask,
            dropout_p=dropout if module.training else 0.0,
            is_causal=False,
            **sdpa_kwargs,
        )
        return out, None

    attn_mod.eager_attention_forward = fused_sdpa_attention_forward
    attn_mod._sparkx_sdpa_patched = True
    return len(layers)


class LoRALinear(nn.Module):
    """Simple native torch LoRA, intentionally keeps .weight accessible for Spark MLP code."""
    def __init__(self, original: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        if not isinstance(original, nn.Linear):
            raise TypeError(f"LoRA expected nn.Linear, got {type(original)}")
        self.base = original
        self.rank = rank
        self.scale = alpha / rank
        self.lora_A = nn.Linear(original.in_features, rank, bias=False, dtype=torch.float32, device=original.weight.device)
        self.lora_B = nn.Linear(rank, original.out_features, bias=False, dtype=torch.float32, device=original.weight.device)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        out = self.base(x)
        delta = self.lora_B(self.lora_A(x.float())) * self.scale
        return out + delta.to(out.dtype)


def attach_lora(base_model: nn.Module, start: int = 4, end: int = 36, rank: int = 8, alpha: float = 16.0,
                attention: bool = True, mlp: bool = True):
    """Attach LoRA once to original physical layers. The same LoRA weights run in every recurrence."""
    changed = []
    layers = base_model.model.layers
    for idx in range(start, min(end, len(layers))):
        layer = layers[idx]
        paths = []
        if attention:
            paths += [(layer.self_attn, 'q_k_v_proj'), (layer.self_attn, 'out_proj')]
            if getattr(layer.self_attn, 'g_proj', None) is not None:
                paths.append((layer.self_attn, 'g_proj'))
        if mlp:
            paths += [(layer.mlp, 'gate_proj'), (layer.mlp, 'up_proj'), (layer.mlp, 'down_proj')]
        for parent, attr in paths:
            module = getattr(parent, attr)
            if isinstance(module, LoRALinear):
                continue
            setattr(parent, attr, LoRALinear(module, rank=rank, alpha=alpha))
            changed.append(f'{idx}:{attr}')
    return changed


class HashedBigramEngram(nn.Module):
    """Memory-Tiered MoE-Engram (DeepSeek Conditional Memory + Shared/Routed PEER Experts).

    Combines three cooperating memory tiers with O(1) / sparse Top-K routing:
    1. Tier 1 (Lexical N-Gram Tier): Multi-head causal modular hashing over 2-gram (t-1, t),
       3-gram (t-2, t-1, t), and skip-2-gram (t-2, t) with O(1) embedding table lookup.
    2. Tier 2 (Hot Shared Knowledge Tier): DeepSeekMoE-style always-active compact low-rank
       shared expert (`shared_down` -> SiLU -> `shared_up`) capturing universal reasoning primitives.
    3. Tier 3 (Cold/Domain Sparse Routed Memory Experts): MoE + PEER/UltraMemV2-style bank of
       `num_experts` memory experts (`expert_down`, `expert_up`), activating only Top-K (`top_k=2`)
       experts per token with pass-conditioned routing (`pass_router_bias`) and auxiliary-loss-free
       load balancing (`expert_bias`, `expert_load_ema`).
    4. Context-Aware Compatibility Gate & Zero-Init Projection: Modulates fused tiered memory by
       `2 * sigmoid(ctx_gate(h_t))` and projects via zero-initialized `project.weight` so Step-0
       output is strictly 0.0 (preserving exact backbone behavior at initialization).
    """
    def __init__(self, hidden: int, buckets: int = 16384, dim: int = 64,
                 num_experts: int = 8, top_k: int = 2, expert_rank: int = 16,
                 shared_rank: int = 16, max_passes: int = 4, rmsnorm_eps: float = 1e-6):
        super().__init__()
        self.hidden = hidden
        self.buckets = buckets
        self.dim = dim
        self.num_experts = max(1, int(num_experts))
        self.top_k = min(max(1, int(top_k)), self.num_experts)
        self.expert_rank = max(1, int(expert_rank))
        self.shared_rank = max(1, int(shared_rank))
        self.max_passes = max(1, int(max_passes))
        self.rmsnorm_eps = rmsnorm_eps

        # Tier 1: O(1) Multi-Scale Causal N-Gram Lookup Tables
        self.emb = nn.Embedding(buckets, dim, dtype=torch.float32)
        self.emb_tri = nn.Embedding(buckets, dim, dtype=torch.float32)
        self.emb_skip = nn.Embedding(buckets, dim, dtype=torch.float32)

        # Tier 2: Hot Shared Knowledge Expert (always active, low-rank SwiGLU/SiLU bottleneck)
        self.shared_down = nn.Linear(hidden, self.shared_rank, bias=False, dtype=torch.float32)
        self.shared_up = nn.Linear(self.shared_rank, dim, bias=False, dtype=torch.float32)

        # Tier 3: Sparse Routed Domain Memory Experts (MoE + PEER/UltraMemV2 value expansion)
        self.router = nn.Linear(hidden, self.num_experts, bias=False, dtype=torch.float32)
        self.pass_router_bias = nn.Parameter(torch.zeros(self.max_passes, self.num_experts, dtype=torch.float32))
        self.register_buffer('expert_bias', torch.zeros(self.num_experts, dtype=torch.float32))
        self.register_buffer('expert_load_ema', torch.full((self.num_experts,), 1.0 / self.num_experts, dtype=torch.float32))
        self.query_proj = nn.Linear(hidden, dim, bias=False, dtype=torch.float32)
        self.expert_down = nn.Parameter(torch.empty(self.num_experts, self.expert_rank, dim, dtype=torch.float32))
        self.expert_up = nn.Parameter(torch.empty(self.num_experts, self.expert_rank, dim, dtype=torch.float32))

        # DeepSeek Engram Context-Aware Compatibility Gate & Zero-Init Backbone Projection
        self.ctx_gate = nn.Linear(hidden, dim, bias=True, dtype=torch.float32)
        self.project = nn.Linear(dim, hidden, bias=False, dtype=torch.float32)
        self.gain = nn.Parameter(torch.tensor(-2.0, dtype=torch.float32))

        nn.init.normal_(self.emb.weight, std=0.02)
        nn.init.normal_(self.emb_tri.weight, std=0.02)
        nn.init.normal_(self.emb_skip.weight, std=0.02)
        nn.init.kaiming_uniform_(self.shared_down.weight, a=math.sqrt(5))
        nn.init.normal_(self.shared_up.weight, std=0.02)
        nn.init.normal_(self.router.weight, std=0.01)
        nn.init.kaiming_uniform_(self.query_proj.weight, a=math.sqrt(5))
        nn.init.normal_(self.expert_down, std=0.02)
        nn.init.normal_(self.expert_up, std=0.02)
        nn.init.zeros_(self.ctx_gate.weight)
        nn.init.zeros_(self.ctx_gate.bias)
        nn.init.zeros_(self.project.weight)

    @staticmethod
    def compute_ngram_indices(
        input_ids: torch.LongTensor,
        buckets: int,
        prev_ids: Optional[torch.LongTensor] = None,
    ) -> tuple[torch.LongTensor, torch.LongTensor, torch.LongTensor]:
        # Deterministic causal hashing over (t-2, t-1, t); zero access to future tokens.
        bsz, seq_len = input_ids.shape
        if prev_ids is None:
            z1 = torch.zeros_like(input_ids[:, :1])
            prev1 = torch.cat((z1, input_ids[:, :-1]), dim=1)
            z2 = torch.zeros_like(input_ids[:, :2]) if seq_len >= 2 else torch.zeros_like(input_ids)
            prev2 = torch.cat((z2, input_ids[:, :-2]), dim=1) if seq_len >= 2 else torch.zeros_like(input_ids)
        else:
            last1 = prev_ids[:, -1:]
            prev1 = torch.cat((last1, input_ids[:, :-1]), dim=1) if seq_len > 1 else last1
            if prev_ids.shape[1] >= 2:
                last2 = prev_ids[:, -2:-1]
            else:
                last2 = torch.zeros_like(last1)
            if seq_len == 1:
                prev2 = last2
            else:
                prev2 = torch.cat((last2, last1, input_ids[:, :-2]), dim=1)
        idx_bi = ((prev1 * 1000003 + input_ids * 9176 + 29) % buckets).long()
        idx_tri = ((prev2 * 2000003 + prev1 * 1000033 + input_ids * 19391 + 73) % buckets).long()
        idx_skip = ((prev2 * 1500007 + input_ids * 31337 + 53) % buckets).long()
        return idx_bi, idx_tri, idx_skip

    def forward(self, input_ids: torch.LongTensor, dtype: torch.dtype,
                prev_ids: Optional[torch.LongTensor] = None,
                hidden_states: Optional[torch.Tensor] = None,
                pass_idx: int = 0,
                ngram_indices: Optional[tuple[torch.LongTensor, torch.LongTensor, torch.LongTensor]] = None):
        if ngram_indices is None:
            idx_bi, idx_tri, idx_skip = self.compute_ngram_indices(input_ids, self.buckets, prev_ids=prev_ids)
        else:
            idx_bi, idx_tri, idx_skip = ngram_indices

        with torch.autocast(device_type=input_ids.device.type, enabled=False):
            # Tier 1: O(1) Lexical N-Gram Lookup
            mem_lex = self.emb(idx_bi) + 0.5 * self.emb_tri(idx_tri) + 0.5 * self.emb_skip(idx_skip)

            if hidden_states is not None:
                h_f32 = hidden_states.float()
                h_norm = h_f32 * torch.rsqrt(h_f32.pow(2).mean(dim=-1, keepdim=True) + self.rmsnorm_eps)

                # Tier 2: Hot Shared Knowledge Expert
                mem_shared = self.shared_up(F.silu(self.shared_down(h_norm)))

                # Tier 3: Cold/Domain Sparse Routed Memory Experts (Top-K MoE + PEER)
                p_idx = min(max(0, int(pass_idx)), self.max_passes - 1)
                router_logits = self.router(h_norm) + self.pass_router_bias[p_idx].view(1, 1, -1)
                routing_probs = F.softmax(router_logits, dim=-1)
                _, top_idx = torch.topk(routing_probs + self.expert_bias.view(1, 1, -1), k=self.top_k, dim=-1)
                top_w = torch.gather(routing_probs, dim=-1, index=top_idx)
                top_w = top_w / (top_w.sum(dim=-1, keepdim=True) + 1e-6)

                if self.training:
                    with torch.no_grad():
                        one_hot = F.one_hot(top_idx, num_classes=self.num_experts).float()
                        freq = one_hot.mean(dim=(0, 1, 2))
                        self.expert_load_ema.mul_(0.95).add_(freq, alpha=0.05)
                        target_freq = 1.0 / float(self.num_experts)
                        self.expert_bias.add_(torch.sign(target_freq - self.expert_load_ema) * 1e-3).clamp_(-0.5, 0.5)

                q = self.query_proj(h_norm) + mem_lex
                p_k = self.expert_down[top_idx]  # [B, T, K, R, dim]
                v_k = self.expert_up[top_idx]    # [B, T, K, R, dim]
                act = F.silu((p_k * q.unsqueeze(2).unsqueeze(3)).sum(dim=-1))  # [B, T, K, R]
                exp_out = (act.unsqueeze(-1) * v_k).sum(dim=-2)                # [B, T, K, dim]
                mem_routed = (top_w.unsqueeze(-1) * exp_out).sum(dim=-2)       # [B, T, dim]

                mem = mem_lex + mem_shared + mem_routed
                mem = mem * torch.rsqrt(mem.pow(2).mean(dim=-1, keepdim=True) + self.rmsnorm_eps)
                gate = 2.0 * torch.sigmoid(self.ctx_gate(h_norm))
                mem = mem * gate
            else:
                mem = mem_lex * torch.rsqrt(mem_lex.pow(2).mean(dim=-1, keepdim=True) + self.rmsnorm_eps)

            z = self.project(mem) * torch.sigmoid(self.gain)
        return z.to(dtype)


TieredMoEEngram = HashedBigramEngram


class TwoStreamMHC(nn.Module):
    """Two-stream Manifold-Constrained Hyper-Connections (mHC) with:
    1. Sublayer Dual-mHC (`mhc_attn` + `mhc_ffn`) matching SenseNova-Loop-9B-Preview.
    2. Token-dependent FP32 RMSNorm + zero-initialized linear Sinkhorn routing (`dynamic_proj`).
    3. Pass-indexed routing biases & zero-initialized low-rank deliberation adapters so Pass 0
       and Pass 1+ can specialize despite sharing physical backbone weights.
    4. Exact zero-gated identity initialization (`extra_gain=0`, `ffn_extra_gain=0`) so Pass 2
       is an exact mathematical identity over Pass 1 at step 0.
    """
    def __init__(self, n_streams: int = 2, sinkhorn_steps: int = 20,
                 hidden_size: int = 0, dynamic_routing: bool = True,
                 sublayer_mhc: bool = True, max_passes: int = 4,
                 pass_adapter_rank: int = 16, rmsnorm_eps: float = 1e-6):
        super().__init__()
        if n_streams != 2:
            raise ValueError('Prototype intentionally supports exactly two streams')
        self.n = n_streams
        self.sinkhorn_steps = sinkhorn_steps
        self.hidden_size = hidden_size
        self.dynamic_routing = bool(dynamic_routing and hidden_size > 0)
        self.sublayer_mhc = bool(sublayer_mhc)
        self.max_passes = max(1, int(max_passes))
        self.rmsnorm_eps = rmsnorm_eps

        # Primary / Attention mHC parameters
        self.mix_logits = nn.Parameter(torch.tensor([[4., -4.], [-4., 4.]], dtype=torch.float32))
        self.read_logits = nn.Parameter(torch.tensor([5., -5.], dtype=torch.float32))
        self.write_logits = nn.Parameter(torch.tensor([5., -5.], dtype=torch.float32))
        self.extra_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))

        # Sublayer FFN mHC parameters (used when layer exposes input_layernorm & post_attention_layernorm)
        self.ffn_mix_logits = nn.Parameter(torch.tensor([[4., -4.], [-4., 4.]], dtype=torch.float32))
        self.ffn_read_logits = nn.Parameter(torch.tensor([5., -5.], dtype=torch.float32))
        self.ffn_write_logits = nn.Parameter(torch.tensor([5., -5.], dtype=torch.float32))
        self.ffn_extra_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))

        # Pass-specific routing bias: [max_passes, 2 (attn/ffn), 8 (2 read + 2 write + 4 mix)]
        self.pass_routing_bias = nn.Parameter(torch.zeros(self.max_passes, 2, 2 * n_streams + n_streams * n_streams, dtype=torch.float32))

        if self.dynamic_routing:
            out_features = 2 * n_streams + n_streams * n_streams
            self.dynamic_proj = nn.Linear(n_streams * hidden_size, out_features, bias=False, dtype=torch.float32)
            self.ffn_dynamic_proj = nn.Linear(n_streams * hidden_size, out_features, bias=False, dtype=torch.float32)
            nn.init.zeros_(self.dynamic_proj.weight)
            nn.init.zeros_(self.ffn_dynamic_proj.weight)
        else:
            self.register_parameter('dynamic_proj', None)
            self.register_parameter('ffn_dynamic_proj', None)

        # Pass-conditioned low-rank deliberation adapter (zero-initialized up-projection)
        if hidden_size > 0 and pass_adapter_rank > 0:
            self.pass_down = nn.ModuleList([
                nn.Linear(hidden_size, pass_adapter_rank, bias=False, dtype=torch.float32)
                for _ in range(self.max_passes)
            ])
            self.pass_up = nn.ModuleList([
                nn.Linear(pass_adapter_rank, hidden_size, bias=False, dtype=torch.float32)
                for _ in range(self.max_passes)
            ])
            for d, u in zip(self.pass_down, self.pass_up):
                nn.init.kaiming_uniform_(d.weight, a=math.sqrt(5))
                nn.init.zeros_(u.weight)
        else:
            self.pass_down = None
            self.pass_up = None

    def mixing_matrix(self, dyn_mix: Optional[torch.Tensor] = None, base_logits: Optional[torch.Tensor] = None):
        # Sinkhorn in log space for stability, supporting both [2, 2] and [B, T, 2, 2].
        x = (self.mix_logits if base_logits is None else base_logits).float()
        if dyn_mix is not None:
            x = x + dyn_mix.float()
        for _ in range(self.sinkhorn_steps):
            x = x - torch.logsumexp(x, dim=-1, keepdim=True)
            x = x - torch.logsumexp(x, dim=-2, keepdim=True)
        return torch.exp(x)

    def _compute_routing(self, streams_f32: torch.Tensor, pass_idx: int = 0, is_ffn: bool = False):
        # streams_f32: [2, B, T, D]
        p_idx = min(max(0, int(pass_idx)), self.max_passes - 1)
        sub_idx = 1 if is_ffn else 0
        base_r = self.ffn_read_logits if is_ffn else self.read_logits
        base_w = self.ffn_write_logits if is_ffn else self.write_logits
        base_m = self.ffn_mix_logits if is_ffn else self.mix_logits
        proj = self.ffn_dynamic_proj if is_ffn else self.dynamic_proj
        p_bias = self.pass_routing_bias[p_idx, sub_idx].float()

        if not self.dynamic_routing or proj is None:
            r_log = base_r.float() + p_bias[:self.n]
            w_log = base_w.float() + p_bias[self.n:2 * self.n]
            m_log = base_m.float() + p_bias[2 * self.n:].reshape(self.n, self.n)
            rw = r_log.softmax(dim=0)[:, None, None, None]
            ww = w_log.softmax(dim=0)[:, None, None, None]
            mat = self.mixing_matrix(base_logits=m_log)[:, :, None, None, None]
            return rw, ww, mat

        with torch.autocast(device_type=streams_f32.device.type, enabled=False):
            _, bsz, seq_len, dim = streams_f32.shape
            flat = streams_f32.permute(1, 2, 0, 3).reshape(bsz, seq_len, self.n * dim)
            normed = flat * torch.rsqrt(flat.pow(2).mean(dim=-1, keepdim=True) + self.rmsnorm_eps)
            dyn = proj(normed) + p_bias.view(1, 1, -1)  # [B, T, 8]
            dyn_read = dyn[..., :self.n]
            dyn_write = dyn[..., self.n:2 * self.n]
            dyn_mix = dyn[..., 2 * self.n:].reshape(bsz, seq_len, self.n, self.n)
            rw = (base_r.float() + dyn_read).softmax(dim=-1).permute(2, 0, 1).unsqueeze(-1)
            ww = (base_w.float() + dyn_write).softmax(dim=-1).permute(2, 0, 1).unsqueeze(-1)
            mat = self.mixing_matrix(dyn_mix, base_logits=base_m).permute(2, 3, 0, 1).unsqueeze(-1)
            return rw, ww, mat

    def _apply_pass_adapter(self, x_f32: torch.Tensor, pass_idx: int) -> torch.Tensor:
        if self.pass_down is None or self.pass_up is None:
            return torch.zeros_like(x_f32)
        p_idx = min(max(0, int(pass_idx)), self.max_passes - 1)
        with torch.autocast(device_type=x_f32.device.type, enabled=False):
            normed = x_f32 * torch.rsqrt(x_f32.pow(2).mean(dim=-1, keepdim=True) + self.rmsnorm_eps)
            return self.pass_up[p_idx](F.silu(self.pass_down[p_idx](normed)))

    def forward(self, streams: torch.Tensor, layer: nn.Module, rope, mask,
                extra: bool = False, pass_idx: int = 0,
                past_key_values=None, cache_position=None):
        # streams: [2, batch, seq, hidden]. Layer is shared across passes.
        streams_f32 = streams.float()
        identity = torch.eye(self.n, dtype=torch.float32, device=streams.device)[:, :, None, None, None]

        # Sublayer Dual-mHC path when wrapping a real Spark2_5DecoderLayer
        if self.sublayer_mhc and hasattr(layer, 'input_layernorm') and hasattr(layer, 'post_attention_layernorm'):
            # 1. Attention sublayer mHC
            rw_a, ww_a, mat_a = self._compute_routing(streams_f32, pass_idx=pass_idx, is_ffn=False)
            mixed_a = (rw_a * streams_f32).sum(dim=0)
            target_dtype = layer.mlp.gate_proj.weight.dtype
            normed_a = layer.input_layernorm(mixed_a.to(streams.dtype)).to(target_dtype)
            attn_out, _ = layer.self_attn(
                hidden_states=normed_a,
                position_embeddings=rope,
                attention_mask=mask,
                past_key_values=past_key_values,
                cache_position=cache_position,
            )
            gain_a = torch.tanh(self.extra_gain.float()) if extra else 1.0
            eff_mat_a = identity + gain_a * (mat_a - identity)
            streams_mid = (eff_mat_a * streams_f32.unsqueeze(0)).sum(dim=1) + gain_a * ww_a * attn_out.float().unsqueeze(0)

            # 2. FFN sublayer mHC + Pass-conditioned deliberation adapter
            rw_f, ww_f, mat_f = self._compute_routing(streams_mid, pass_idx=pass_idx, is_ffn=True)
            mixed_f = (rw_f * streams_mid).sum(dim=0)
            normed_f = layer.post_attention_layernorm(mixed_f.to(streams.dtype)).to(target_dtype)
            mlp_out = layer.mlp(normed_f).float() + self._apply_pass_adapter(mixed_f, pass_idx)
            gain_f = torch.tanh(self.ffn_extra_gain.float()) if extra else 1.0
            eff_mat_f = identity + gain_f * (mat_f - identity)
            out = (eff_mat_f * streams_mid.unsqueeze(0)).sum(dim=1) + gain_f * ww_f * mlp_out.unsqueeze(0)
            return out.to(streams.dtype)

        # Fallback single-wrapper mHC path (e.g. simplified test doubles without separate layernorms)
        rw, ww, mat = self._compute_routing(streams_f32, pass_idx=pass_idx, is_ffn=False)
        mixed_input = (rw * streams_f32).sum(dim=0).to(streams.dtype)
        updated = layer(
            mixed_input, position_embeddings=rope, attention_mask=mask,
            past_key_values=past_key_values, cache_position=cache_position,
        )
        delta = updated.float() - mixed_input.float() + self._apply_pass_adapter(mixed_input.float(), pass_idx)
        gain = torch.tanh(self.extra_gain.float()) if extra else 1.0
        eff_mat = identity + gain * (mat - identity)
        routed = (eff_mat * streams_f32.unsqueeze(0)).sum(dim=1)
        out = routed + gain * ww * delta.unsqueeze(0)
        return out.to(streams.dtype)


def fused_masked_cross_entropy(hidden_states: torch.Tensor, head_weight: torch.Tensor,
                               labels: torch.Tensor, chunk_tokens: int = 512) -> torch.Tensor:
    """Unsloth-style memory-efficient masked cross-entropy over vocabulary size 167,168.

    Gathers only supervised positions (`shift_labels != -100`) and evaluates the FP32
    vocabulary projection in small token chunks under gradient checkpointing so the
    massive [B, T, 167168] FP32 logits tensor is never materialized in VRAM.
    """
    shift_h = hidden_states[:, :-1].contiguous().reshape(-1, hidden_states.shape[-1])
    shift_y = labels[:, 1:].contiguous().reshape(-1).to(shift_h.device)
    valid_mask = shift_y != -100
    if not torch.any(valid_mask):
        return hidden_states.sum() * 0.0
    valid_h = shift_h[valid_mask]
    valid_y = shift_y[valid_mask]
    n_valid = valid_y.numel()
    if n_valid <= chunk_tokens:
        logits = F.linear(valid_h.to(head_weight.dtype), head_weight).float()
        return F.cross_entropy(logits, valid_y, reduction='mean')

    def _chunk_ce(h_chunk: torch.Tensor, y_chunk: torch.Tensor) -> torch.Tensor:
        lg = F.linear(h_chunk.to(head_weight.dtype), head_weight).float()
        return F.cross_entropy(lg, y_chunk, reduction='sum')

    total_loss = torch.zeros((), dtype=torch.float32, device=hidden_states.device)
    use_ckpt = valid_h.requires_grad or head_weight.requires_grad
    for start_idx in range(0, n_valid, chunk_tokens):
        h_c = valid_h[start_idx:start_idx + chunk_tokens]
        y_c = valid_y[start_idx:start_idx + chunk_tokens]
        if use_ckpt:
            total_loss = total_loss + checkpoint(_chunk_ce, h_c, y_c, use_reentrant=False)
        else:
            total_loss = total_loss + _chunk_ce(h_c, y_c)
    return total_loss / float(n_valid)


class SparkXLoop(nn.Module):
    """Wrap original Spark2_5ForCausalLM and execute real layers with shared depth recurrence.

    Reproduces the upstream model's masks, type-dependent RoPE and norm/head path.
    Supports both cache-free training/verification (generate_slow) and per-visit
    KV-cached looped decoding (generate_cached via _VisitCache).
    """
    def __init__(self, base: nn.Module, settings: Optional[LoopSettings] = None,
                 strict: bool = True, patch_sdpa: bool = True):
        super().__init__()
        self.base = base
        self.settings = settings or LoopSettings()
        self.n_layers = len(self.base.model.layers)
        self.width = int(self.base.config.hidden_size)
        if strict and (self.n_layers != 36 or self.width != 2560):
            raise RuntimeError(f'Expected Spark 36x2560, got {self.n_layers}x{self.width}')
        if self.settings.start < 0 or self.settings.end > self.n_layers:
            raise ValueError('Bad recurrent range')
        if patch_sdpa:
            patch_spark_attention_for_long_context(self.base)
        self.controllers = nn.ModuleDict({
            str(i): TwoStreamMHC(
                self.settings.mhc_streams,
                hidden_size=self.width,
                dynamic_routing=self.settings.dynamic_mhc,
                sublayer_mhc=self.settings.sublayer_mhc,
                max_passes=self.settings.max_passes,
                pass_adapter_rank=self.settings.pass_adapter_rank,
            )
            for i in range(self.settings.start, self.settings.end)
        })
        def _make_engram():
            return HashedBigramEngram(
                self.width,
                buckets=self.settings.eng_buckets,
                dim=self.settings.eng_dim,
                num_experts=self.settings.eng_experts,
                top_k=self.settings.eng_top_k,
                expert_rank=self.settings.eng_expert_rank,
                shared_rank=self.settings.eng_shared_rank,
                max_passes=self.settings.max_passes,
            )

        self.engram = _make_engram()
        stride = max(0, int(self.settings.backbone_engram_stride))
        if stride > 0:
            self.backbone_engrams = nn.ModuleDict({
                str(i): _make_engram()
                for i in range(self.settings.start, self.settings.end)
                if (i - self.settings.start + 1) % stride == 0
            })
        else:
            self.backbone_engrams = nn.ModuleDict()
        self.enable_checkpoint = self.settings.checkpoint_layers
        self.offload_checkpoint_to_cpu = self.settings.offload_checkpoint_to_cpu
        self.loop_start, self.loop_end, self.loop_passes = self.settings.start, self.settings.end, self.settings.passes
        self.mhc_enabled = self.settings.use_mhc
        self.engram_enabled = self.settings.use_engram
        self._validate_loop()

    def _validate_loop(self):
        if not (0 <= self.loop_start < self.loop_end <= self.n_layers):
            raise ValueError('Invalid loop boundaries')
        if (self.loop_start % 4 or self.loop_end % 4):
            raise ValueError('Loop boundaries must align to Spark groups of four')
        if self.loop_passes < 1:
            raise ValueError('loop_passes must be >= 1')

    @property
    def execution_plan(self) -> list[int]:
        return (
            list(range(0, self.loop_start))
            + list(range(self.loop_start, self.loop_end)) * self.loop_passes
            + list(range(self.loop_end, self.n_layers))
        )

    @property
    def total_virtual_layers(self) -> int:
        return len(self.execution_plan)

    def _make_loop_cache_config(self):
        cfg = copy.deepcopy(self.base.config)
        plan = self.execution_plan
        cfg.num_hidden_layers = len(plan)
        if hasattr(cfg, 'layer_types') and cfg.layer_types:
            cfg.layer_types = [cfg.layer_types[i] for i in plan]
        return cfg

    def _checkpoint_context(self):
        if self.training and self.enable_checkpoint and self.offload_checkpoint_to_cpu:
            return torch.autograd.graph.save_on_cpu(pin_memory=True)
        return nullcontext()

    def set_loop(self, start: int, end: int, passes: int, use_mhc: bool = True, use_engram: bool = True):
        self.loop_start, self.loop_end, self.loop_passes = start, end, passes
        self.mhc_enabled, self.engram_enabled = use_mhc, use_engram
        self._validate_loop()
        for idx in range(start, end):
            if str(idx) not in self.controllers:
                raise RuntimeError(f'Controller for layer {idx} was not created: use initial superset')

    def _forward_context(self, embeds, attention_mask, past_key_values=None, cache_position=None):
        # Keep the same mask and rotary logic as Spark2_5Model.forward.
        model = self.base.model
        cfg = self.base.config
        device = embeds.device
        if cache_position is None:
            past_seen = past_key_values.get_seq_length(0) if past_key_values is not None else 0
            cache_position = torch.arange(past_seen, past_seen + embeds.shape[1], device=device)
        position_ids = cache_position.unsqueeze(0)
        # Test doubles can supply a minimal context without importing Transformers.
        if hasattr(model, 'make_loop_context'):
            return model.make_loop_context(embeds, attention_mask), cache_position
        from transformers.masking_utils import create_causal_mask, create_sliding_window_causal_mask
        mask_cfg = self._make_loop_cache_config() if past_key_values is not None else cfg
        kwargs = dict(config=mask_cfg, input_embeds=embeds, attention_mask=attention_mask,
                      cache_position=cache_position, past_key_values=past_key_values, position_ids=position_ids)
        masks = {'full_attention': create_causal_mask(**kwargs)}
        if getattr(model, 'has_sliding_layers', False):
            masks['sliding_attention'] = create_sliding_window_causal_mask(**kwargs)
        implementation = importlib.import_module(type(model.layers[0].self_attn).__module__)
        rope = {}
        for layer_type in set(cfg.layer_types):
            rope[layer_type] = implementation.compute_rope_cos_sin(
                cache_position, cfg.head_dim, cfg.get_rope_theta(layer_type),
                partial_rotary_factor=cfg.get_partial_rotary_factor(layer_type), device=device,
            )
        return (masks, rope), cache_position

    def _apply_layer(self, idx, h, masks, ropes, past_key_values=None, cache_position=None, virtual_idx: Optional[int] = None):
        layer = self.base.model.layers[idx]
        kind = layer.layer_type
        rp, mk = ropes[kind], masks[kind]
        v_cache = _VisitCache(past_key_values, idx, virtual_idx if virtual_idx is not None else idx) if past_key_values is not None else None
        def step(x, _layer=layer, _rp=rp, _mk=mk, _vc=v_cache, _cp=cache_position):
            return _layer(x, position_embeddings=_rp, attention_mask=_mk,
                          past_key_values=_vc, cache_position=_cp)
        if v_cache is None and self.training and self.enable_checkpoint and (
            h.requires_grad or any(p.requires_grad for p in layer.parameters())
        ):
            with self._checkpoint_context():
                return checkpoint(step, h, use_reentrant=False)
        return step(h)

    def forward_hidden(self, input_ids: torch.LongTensor, attention_mask=None,
                       past_key_values=None, cache_position=None,
                       prev_ids: Optional[torch.LongTensor] = None) -> torch.Tensor:
        if input_ids.ndim != 2:
            raise ValueError('Expected input_ids shape [batch, sequence]')
        embeds = self.base.model.embedding(input_ids)
        (masks, ropes), cache_position = self._forward_context(
            embeds, attention_mask, past_key_values=past_key_values, cache_position=cache_position
        )
        h = embeds.float()  # matches upstream Spark 2.5 model forward
        virtual_idx = 0
        ngram_indices = (
            self.engram.compute_ngram_indices(input_ids, self.settings.eng_buckets, prev_ids=prev_ids)
            if self.engram_enabled else None
        )

        # Original prelude layers, unchanged.
        for idx in range(self.loop_start):
            h = self._apply_layer(idx, h, masks, ropes, past_key_values=past_key_values,
                                  cache_position=cache_position, virtual_idx=virtual_idx)
            virtual_idx += 1
        # Memory-Tiered MoE-Engram inserted after prelude; zero-initialized project keeps step-0 parity exact.
        if self.engram_enabled:
            h = h + self.engram(
                input_ids, h.dtype, prev_ids=prev_ids, hidden_states=h,
                pass_idx=0, ngram_indices=ngram_indices,
            )

        if not self.mhc_enabled:
            # Direct recurrence negative control.
            for p in range(self.loop_passes):
                for idx in range(self.loop_start, self.loop_end):
                    h = self._apply_layer(idx, h, masks, ropes, past_key_values=past_key_values,
                                          cache_position=cache_position, virtual_idx=virtual_idx)
                    if self.engram_enabled and str(idx) in self.backbone_engrams:
                        eng_delta = self.backbone_engrams[str(idx)](
                            input_ids, h.dtype, prev_ids=prev_ids, hidden_states=h,
                            pass_idx=p, ngram_indices=ngram_indices,
                        )
                        if p == 0:
                            h = h + eng_delta
                    virtual_idx += 1
        else:
            streams = torch.stack((h, h), dim=0)
            for p in range(self.loop_passes):
                for idx in range(self.loop_start, self.loop_end):
                    kind = self.base.model.layers[idx].layer_type
                    layer = self.base.model.layers[idx]
                    ctrl = self.controllers[str(idx)]
                    rp, mk = ropes[kind], masks[kind]
                    v_cache = _VisitCache(past_key_values, idx, virtual_idx) if past_key_values is not None else None
                    eng_mod = (
                        self.backbone_engrams[str(idx)]
                        if (self.engram_enabled and str(idx) in self.backbone_engrams)
                        else None
                    )
                    virtual_idx += 1
                    # Capture immutable locals, not a changing Python loop variable.
                    def step(s, _ctrl=ctrl, _layer=layer, _rp=rp, _mk=mk, _extra=(p > 0), _p=p, _vc=v_cache, _cp=cache_position, _eng=eng_mod):
                        s_out = _ctrl(s, _layer, _rp, _mk, extra=_extra, pass_idx=_p,
                                      past_key_values=_vc, cache_position=_cp)
                        if _eng is not None:
                            eng_delta = _eng(
                                input_ids, s_out.dtype, prev_ids=prev_ids, hidden_states=s_out[0],
                                pass_idx=_p, ngram_indices=ngram_indices,
                            )
                            if _extra:
                                eng_delta = eng_delta * torch.tanh(_ctrl.ffn_extra_gain.float()).to(s_out.dtype)
                            s_out = torch.stack((s_out[0] + eng_delta, s_out[1]), dim=0)
                        return s_out
                    if v_cache is None and self.training and self.enable_checkpoint:
                        with self._checkpoint_context():
                            streams = checkpoint(step, streams, use_reentrant=False)
                    else:
                        streams = step(streams)
            h = streams[0]

        for idx in range(self.loop_end, self.n_layers):
            h = self._apply_layer(idx, h, masks, ropes, past_key_values=past_key_values,
                                  cache_position=cache_position, virtual_idx=virtual_idx)
            virtual_idx += 1
        return self.base.model.norm(h).to(self.base.model.embedding.weight.dtype)

    def _head_weight(self) -> torch.Tensor:
        if getattr(self.base.config, 'tie_word_embeddings', False):
            return self.base.model.embedding.weight
        return self.base.lm_head.weight

    def forward_logits(self, input_ids: torch.LongTensor, attention_mask=None,
                       past_key_values=None, cache_position=None,
                       prev_ids: Optional[torch.LongTensor] = None):
        h = self.forward_hidden(
            input_ids, attention_mask=attention_mask,
            past_key_values=past_key_values, cache_position=cache_position,
            prev_ids=prev_ids,
        )
        if getattr(self.base.config, 'tie_word_embeddings', False):
            return F.linear(h, self.base.model.embedding.weight)
        return self.base.lm_head(h)

    def forward(self, input_ids, labels=None, attention_mask=None, return_logits: bool = False):
        if labels is None:
            logits = self.forward_logits(input_ids, attention_mask)
            return {'logits': logits}
        if input_ids.shape[1] < 2:
            raise ValueError('Need at least 2 tokens for next-token objective')
        h = self.forward_hidden(input_ids, attention_mask=attention_mask)
        loss = fused_masked_cross_entropy(
            h, self._head_weight(), labels, chunk_tokens=self.settings.ce_chunk_tokens
        )
        if return_logits:
            logits = F.linear(h, self._head_weight()) if getattr(self.base.config, 'tie_word_embeddings', False) else self.base.lm_head(h)
            return {'logits': logits, 'loss': loss}
        return {'logits': None, 'loss': loss}

    @torch.no_grad()
    def generate_slow(self, input_ids, max_new_tokens=24, eos_token_id=None):
        """Correct full-recompute greedy looped generation. Slow by design; no KV cache."""
        prev_train = self.training
        self.eval()
        out = input_ids
        try:
            for _ in range(max_new_tokens):
                next_id = self.forward_logits(out)[:, -1].argmax(-1, keepdim=True)
                out = torch.cat((out, next_id), dim=-1)
                if eos_token_id is not None and torch.all(next_id == eos_token_id):
                    break
        finally:
            self.train(prev_train)
        return out

    @torch.no_grad()
    def generate_cached(self, input_ids, max_new_tokens=128, eos_token_id=None):
        """Fast per-visit KV-cached greedy looped generation using _VisitCache."""
        from transformers.cache_utils import DynamicCache
        prev_train = self.training
        self.eval()
        try:
            cache = DynamicCache(config=self._make_loop_cache_config())
        except TypeError:
            cache = DynamicCache()
        out = input_ids
        try:
            # Prefill full prompt
            logits = self.forward_logits(out, past_key_values=cache)
            next_id = logits[:, -1].argmax(-1, keepdim=True)
            prev_tok = out[:, -2:]
            out = torch.cat((out, next_id), dim=-1)
            if eos_token_id is not None and torch.all(next_id == eos_token_id):
                return out
            for _ in range(max_new_tokens - 1):
                logits = self.forward_logits(next_id, past_key_values=cache, prev_ids=prev_tok)
                prev_tok = out[:, -2:]
                next_id = logits[:, -1].argmax(-1, keepdim=True)
                out = torch.cat((out, next_id), dim=-1)
                if eos_token_id is not None and torch.all(next_id == eos_token_id):
                    break
        finally:
            self.train(prev_train)
        return out


def set_trainables(model: SparkXLoop, full_layer_ids=(), full_finetune: bool = False,
                   train_embeddings: bool = False):
    """Configure trainable parameters for either Full Finetuning (all 36 layers + mHC + Engram)
    or Hybrid LoRA + selective backbone layers.
    """
    if full_finetune:
        full_layer_ids = set(range(model.n_layers))
    else:
        full_layer_ids = set(full_layer_ids)
    for p in model.base.parameters():
        p.requires_grad_(False)
    if train_embeddings:
        for p in model.base.model.embedding.parameters():
            p.requires_grad_(True)
        for p in model.base.model.norm.parameters():
            p.requires_grad_(True)
        if not getattr(model.base.config, 'tie_word_embeddings', False) and hasattr(model.base, 'lm_head'):
            for p in model.base.lm_head.parameters():
                p.requires_grad_(True)
    elif full_finetune:
        for p in model.base.model.norm.parameters():
            p.requires_grad_(True)
    for mod in model.modules():
        if isinstance(mod, LoRALinear):
            for p in list(mod.lora_A.parameters()) + list(mod.lora_B.parameters()):
                p.requires_grad_(True)
    for p in model.controllers.parameters():
        p.requires_grad_(True)
    for p in model.engram.parameters():
        p.requires_grad_(True)
    for p in model.backbone_engrams.parameters():
        p.requires_grad_(True)
    for idx in full_layer_ids:
        if not (0 <= idx < model.n_layers):
            raise ValueError(f'Invalid full layer {idx}')
        for name, p in model.base.model.layers[idx].named_parameters():
            if not (name.startswith('self_attn.q_k_v_proj.lora_') or
                    name.startswith('self_attn.out_proj.lora_') or '.lora_' in name):
                p.requires_grad_(True)
    return {n: p for n, p in model.named_parameters() if p.requires_grad}


def optimizer_groups(model: SparkXLoop, lr_adapter=1e-4, lr_full=1e-5, weight_decay=0.01, lr_mhc=5e-3):
    adapters, routing, backbone = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if (
            (name.startswith('controllers.') and '.pass_' not in name)
            or name == 'engram.gain'
            or (name.startswith('backbone_engrams.') and name.endswith('.gain'))
        ):
            routing.append(p)
        elif name.startswith('base.model.') and '.lora_' not in name:
            backbone.append(p)
        else:
            adapters.append(p)
    return [{'params': adapters, 'lr': lr_adapter, 'weight_decay': weight_decay},
            {'params': routing, 'lr': lr_mhc, 'weight_decay': 0.0},
            {'params': backbone, 'lr': lr_full, 'weight_decay': weight_decay}]


def build_optimizer(model: SparkXLoop, lr_adapter=1e-4, lr_full=1e-5, weight_decay=0.01,
                    lr_mhc=5e-3, use_8bit: bool = True):
    """Build Unsloth / bitsandbytes PagedAdamW8bit (paging to 130 GB System RAM when needed)
    or fused PyTorch AdamW.
    """
    groups = optimizer_groups(
        model, lr_adapter=lr_adapter, lr_full=lr_full, weight_decay=weight_decay, lr_mhc=lr_mhc
    )
    if use_8bit:
        try:
            import bitsandbytes as bnb
            return bnb.optim.PagedAdamW8bit(groups, betas=(0.9, 0.95), eps=1e-8)
        except Exception:
            pass
    use_fused = torch.cuda.is_available() and 'fused' in inspect.signature(torch.optim.AdamW.__init__).parameters
    return torch.optim.AdamW(groups, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)


def trainable_state(model: SparkXLoop):
    return {name: p.detach().cpu().contiguous() for name, p in model.named_parameters() if p.requires_grad}


def count_params(model: SparkXLoop):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile load_inference.py
"""SparkX-Loop v0.2 inference from a Hugging Face stage export.

Supports both per-visit KV-cached looped decoding (`generate_cached` via `_VisitCache`)
and exact cache-free full-recompute decoding (`generate_slow`), plus Spark X2.6's native
`<think>...</think>` reasoning mode (`ENABLE_THINKING=1`) and 1M-context fused SDPA.

Usage (outside Colab after training):
    pip install torch transformers==4.57.1 accelerate safetensors huggingface_hub
    HF_TOKEN=hf_xxx LOOP_REPO=TassP0lar/Spark-X2.6-4B-Loop-mHC LOOP_STAGE=R24 python load_inference.py
"""
import json
import os
from pathlib import Path
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from sparkx_loop_core import LoopSettings, SparkXLoop, attach_lora

repo=os.environ.get('LOOP_REPO','TassP0lar/Spark-X2.6-4B-Loop-mHC')
stage=os.environ.get('LOOP_STAGE','R24')
token=os.environ.get('HF_TOKEN') or os.environ.get('HF_TOKEM')
enable_thinking=os.environ.get('ENABLE_THINKING','1').lower() not in ('0','false','no')
use_visit_cache=os.environ.get('USE_VISIT_CACHE','1').lower() not in ('0','false','no')
max_new_tokens=int(os.environ.get('MAX_NEW_TOKENS','256' if enable_thinking else '64'))
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU required for default inference script')
root=Path(snapshot_download(repo_id=repo,token=token,allow_patterns=[f'stages/{stage}/*']))
folder=root/'stages'/stage
meta=json.loads((folder/'loop_config.json').read_text())
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base=AutoModelForCausalLM.from_pretrained(meta['base_id'],revision=meta['base_revision'],
    trust_remote_code=True,torch_dtype=dtype,device_map={'':'cuda:0'},token=token,attn_implementation='eager')
tok=AutoTokenizer.from_pretrained(meta['base_id'],revision=meta['base_revision'],trust_remote_code=True,token=token)
base.config.use_cache=False
if meta.get('lora_rank',0)>0:
    attach_lora(base,start=4,end=36,rank=meta['lora_rank'],alpha=meta['lora_alpha'])
settings=LoopSettings(
    start=4,end=28,passes=meta['loop_passes'],
    eng_buckets=meta['engram_buckets'],eng_dim=meta['engram_dim'],
    eng_experts=meta.get('engram_experts',8),
    eng_top_k=meta.get('engram_top_k',2),
    eng_expert_rank=meta.get('engram_expert_rank',16),
    eng_shared_rank=meta.get('engram_shared_rank',16),
    backbone_engram_stride=meta.get('backbone_engram_stride',4),
    checkpoint_layers=False,dynamic_mhc=meta.get('dynamic_mhc',True),
    sublayer_mhc=meta.get('sublayer_mhc',True),
    pass_adapter_rank=meta.get('pass_adapter_rank',16),
)
net=SparkXLoop(base,settings,patch_sdpa=True).to('cuda:0')
state=load_file(str(folder/'trainable.safetensors'),device='cpu')
missing,unexpected=net.load_state_dict(state,strict=False)
assert not unexpected,unexpected
net.set_loop(meta['loop_start'],meta['loop_end'],meta['loop_passes'],
             use_mhc=meta['mhc_enabled'],use_engram=meta['engram_enabled'])
net.eval()
messages=[{'role':'user','content':'Explain why looped depth recurrence with sublayer manifold-constrained hyper-connections improves parameter efficiency and multi-step reasoning.'}]
ids=tok.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
    enable_thinking=enable_thinking,reasoning_effort='high',return_tensors='pt')
if hasattr(ids,'input_ids'): ids=ids.input_ids
elif isinstance(ids,dict): ids=ids['input_ids']
ids=ids.to('cuda:0')
if use_visit_cache:
    out=net.generate_cached(ids,max_new_tokens=max_new_tokens,eos_token_id=tok.eos_token_id)
else:
    out=net.generate_slow(ids,max_new_tokens=max_new_tokens,eos_token_id=tok.eos_token_id)
print(tok.decode(out[0,ids.shape[-1]:],skip_special_tokens=False))


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sparkx_loop_core import (
    LoopSettings, SparkXLoop, attach_lora, set_trainables,
    optimizer_groups, build_optimizer, trainable_state, count_params,
    patch_spark_attention_for_long_context,
)

tok = AutoTokenizer.from_pretrained(CFG['MODEL_ID'], revision=CFG['MODEL_REVISION'],
                                    trust_remote_code=True, token=HF_TOKEN)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    CFG['MODEL_ID'], revision=CFG['MODEL_REVISION'], trust_remote_code=True,
    torch_dtype=DTYPE, low_cpu_mem_usage=True, device_map={'':'cuda:0'},
    token=HF_TOKEN, attn_implementation='eager')
if len(base.model.layers) != 36 or int(base.config.hidden_size) != 2560:
    raise AssertionError('Upstream Spark layout changed; this notebook requires 36 layers / width 2560.')
base.config.use_cache = False
base.eval()
patched_layers = patch_spark_attention_for_long_context(base)
print(f'Patched {patched_layers} Spark2_5Attention layers with 1M-Context Fused SDPA / GQA.')
for p in base.parameters(): p.requires_grad_(False)
changed = attach_lora(base, start=4, end=36, rank=CFG['LORA_RANK'], alpha=CFG['LORA_ALPHA'])
settings = LoopSettings(
    start=4, end=28, passes=2, max_passes=4,
    checkpoint_layers=True,
    offload_checkpoint_to_cpu=CFG['OFFLOAD_CKPT_CPU'],
    eng_buckets=CFG['ENG_BUCKETS'], eng_dim=CFG['ENG_DIM'],
    eng_experts=CFG['ENG_EXPERTS'], eng_top_k=CFG['ENG_TOP_K'],
    eng_expert_rank=CFG['ENG_EXPERT_RANK'], eng_shared_rank=CFG['ENG_SHARED_RANK'],
    backbone_engram_stride=CFG['BACKBONE_ENGRAM_STRIDE'],
    dynamic_mhc=CFG['DYNAMIC_MHC'],
    sublayer_mhc=CFG['SUBLAYER_MHC'],
    pass_adapter_rank=CFG['PASS_ADAPTER_RANK'],
    ce_chunk_tokens=CFG['CE_CHUNK_TOKENS'],
)
net = SparkXLoop(base, settings, patch_sdpa=True).to('cuda:0')
net.engram_enabled = CFG['ENGRAM']
trainables = set_trainables(
    net,
    full_layer_ids=CFG['FULL_LAYERS'],
    full_finetune=(CFG['TRAIN_MODE'] == 'full'),
)
print('Injected LoRA:', len(changed), 'linear modules | parameters total/trainable:', count_params(net))
print('Backbone MoE-Engram anchors:', ['prelude:3'] + [f'core:{k}' for k in net.backbone_engrams.keys()],
      f'| Experts/layer: {CFG["ENG_EXPERTS"]} (Top-{CFG["ENG_TOP_K"]})')
print('Virtual layers at R24 (passes=2):', net.total_virtual_layers,
      '| Sublayer Dual-mHC:', CFG['SUBLAYER_MHC'], '| Dynamic mHC:', CFG['DYNAMIC_MHC'])
print('Trainable tensors:', len(trainables), '| Mode:', CFG['TRAIN_MODE'])


In [ ]:
# Fail-fast interface + 1M-context SDPA parity + Dual-mHC zero-gate identity and causality checks.
net.eval()
ids = tok('Hello, what is 2 + 2?', return_tensors='pt')['input_ids'].to('cuda:0')
with torch.no_grad():
    reference = base(input_ids=ids, use_cache=False).logits
    net.set_loop(4,12,1,use_mhc=False,use_engram=False)
    retrofit_identity = net.forward_logits(ids)
    max_diff=(reference.float()-retrofit_identity.float()).abs().max().item()
print('Disabled-extension upstream parity: max abs logit difference =',max_diff)
if max_diff > 0.025:
    raise AssertionError(f'Original-model parity failed ({max_diff}); verify mask/RoPE/transformers version')
# Verify 2-pass with initial extra_gain=0, ffn_extra_gain=0, dynamic_proj=0 matches 1-pass identically.
with torch.no_grad():
    net.set_loop(4,12,1,use_mhc=True,use_engram=CFG['ENGRAM'])
    pass1_logits = net.forward_logits(ids)
    net.set_loop(4,12,2,use_mhc=True,use_engram=CFG['ENGRAM'])
    extra_logits = net.forward_logits(ids)
    zero_gate_diff = (pass1_logits.float()-extra_logits.float()).abs().max().item()
if zero_gate_diff > 1e-5:
    raise AssertionError(f'Initial 2-pass zero-gate identity failed ({zero_gate_diff})')
# Causal path check: changing final token may not alter earlier token predictions on base or looped wrapper.
with torch.no_grad():
    edited=ids.clone()
    edited[0,-1]=(int(edited[0,-1])+13)%int(base.config.vocab_size)
    original_edited=base(input_ids=edited,use_cache=False).logits
    looped_edited=net.forward_logits(edited)
    earlier_error=max(
        (reference[:,:-1].float()-original_edited[:,:-1].float()).abs().max().item(),
        (extra_logits[:,:-1].float()-looped_edited[:,:-1].float()).abs().max().item(),
    )
if earlier_error>0.025:
    raise AssertionError(f'Causal check failed under current mask/Engram configuration ({earlier_error})')
print('Causal mask, Dual-mHC & Multi-Scale Engram check passed; earlier-token max diff =',earlier_error)
del edited,original_edited,looped_edited,pass1_logits
p = net.controllers['4'].mixing_matrix()
assert (p.sum(0)-1).abs().max()<1e-4 and (p.sum(1)-1).abs().max()<1e-4
dyn_p = net.controllers['4'].mixing_matrix(torch.randn(1, 4, 2, 2, device='cuda:0'))
assert (dyn_p.sum(-1)-1).abs().max()<1e-4 and (dyn_p.sum(-2)-1).abs().max()<1e-4
assert torch.isfinite(extra_logits).all()
print('R8 forward and Dual-mHC Sinkhorn checks passed; max allocated VRAM GiB:',
      round(torch.cuda.max_memory_allocated()/2**30,2))
del reference,retrofit_identity,extra_logits,dyn_p; torch.cuda.empty_cache()

In [ ]:
# Multi-source CoT, Scientific Reasoning & Multi-Turn Chat dataset loader.
from datasets import load_dataset

THINK_RE = re.compile(r'<(?:think|thought|reasoning)>\s*(.*?)\s*</(?:think|thought|reasoning)>\s*(.*)', re.DOTALL | re.IGNORECASE)

def split_inline_thinking(text: str):
    if not text:
        return '', ''
    m = THINK_RE.search(text)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    if '<think>' in text:
        parts = text.split('<think>', 1)[1]
        if '</think>' in parts:
            t, a = parts.split('</think>', 1)
            return t.strip(), a.strip()
    return '', text.strip()

def normalize_reasoning_row(row: dict, source_id: str, family: str = 'reasoning'):
    system_text = str(row.get('system') or '').strip()
    user_text = ''
    thinking_text = ''
    answer_text = ''
    history_turns = []

    top_thinking = str(
        row.get('thinking')
        or row.get('reasoning')
        or row.get('reasoning_content')
        or row.get('step_by_step_trace')
        or row.get('inverted_reasoning')
        or ''
    ).strip()
    if top_thinking:
        t_inner, _ = split_inline_thinking(top_thinking)
        thinking_text = t_inner if t_inner else top_thinking

    # Normalize either OpenAI `messages` or ShareGPT `conversations`
    msgs = row.get('messages') or row.get('conversations')
    if isinstance(msgs, str):
        try:
            msgs = json.loads(msgs)
        except Exception:
            msgs = None
    if isinstance(msgs, list) and msgs:
        parsed_turns = []
        for m in msgs:
            if isinstance(m, str):
                try:
                    m = json.loads(m)
                except Exception:
                    continue
            if not isinstance(m, dict):
                continue
            raw_role = str(m.get('role') or m.get('from') or '').lower().strip()
            content = str(m.get('content') or m.get('value') or '').strip()
            if raw_role in ('system', 'sys') and content and not system_text:
                system_text = content
            elif raw_role in ('user', 'human', 'prompter') and content:
                parsed_turns.append(('user', content, ''))
            elif raw_role in ('assistant', 'gpt', 'model', 'bot') and content:
                m_think = str(
                    m.get('reasoning_content')
                    or m.get('thinking')
                    or m.get('reasoning')
                    or ''
                ).strip()
                parsed_turns.append(('assistant', content, m_think))

        # Find the last assistant turn with a preceding user turn (supports multi-turn chat!)
        for idx in range(len(parsed_turns) - 1, 0, -1):
            role, content, m_think = parsed_turns[idx]
            if role == 'assistant' and parsed_turns[idx - 1][0] == 'user':
                user_text = parsed_turns[idx - 1][1]
                # Keep up to 4 prior dialogue turns as multi-turn context history
                for h_role, h_content, _ in parsed_turns[max(0, idx - 5):idx - 1]:
                    _, clean_h = split_inline_thinking(h_content)
                    history_turns.append({'role': h_role, 'content': clean_h or h_content})
                t_in, a_in = split_inline_thinking(content)
                if m_think and not thinking_text:
                    thinking_text = m_think
                    answer_text = a_in if t_in else content
                elif t_in and not thinking_text:
                    thinking_text = t_in
                    answer_text = a_in
                else:
                    answer_text = a_in if t_in else content
                break

    if not user_text:
        instr = str(row.get('instruction') or '').strip()
        inp = str(row.get('input') or '').strip()
        if instr and inp and instr != inp:
            user_text = f'{instr}\n\n{inp}'
        else:
            user_text = str(
                row.get('prompt')
                or instr
                or row.get('question')
                or inp
                or ''
            ).strip()
    if '<|user|>' in user_text and '<|assistant|>' in user_text:
        u_part, a_part = user_text.split('<|assistant|>', 1)
        user_text = u_part.replace('<|user|>', '').strip()
        if not answer_text:
            answer_text = a_part.strip()

    # Check if top-level `output` has <think> tags (e.g. Jackrong/Kimi-K2.5-Reasoning-1M-Cleaned or nvidia/OpenScienceReasoning-2)
    top_out = str(row.get('output') or '').strip()
    if top_out and not thinking_text and '<think>' in top_out:
        t_out, a_out = split_inline_thinking(top_out)
        if t_out:
            thinking_text = t_out
            if a_out:
                answer_text = a_out

    if not answer_text:
        raw_ans = str(
            row.get('response')
            or row.get('answer')
            or row.get('final_solution')
            or top_out
            or ''
        ).strip()
        t_in, a_in = split_inline_thinking(raw_ans)
        if t_in and not thinking_text:
            thinking_text = t_in
            answer_text = a_in
        else:
            answer_text = raw_ans

    # MegaScience / TextbookReasoning: `answer` is textbook derivation, `reference_answer` is concise conclusion
    ref_ans = str(row.get('reference_answer') or row.get('expected_answer') or '').strip()
    if not thinking_text and family == 'science-reasoning' and ref_ans and len(answer_text) > 120 and ref_ans != answer_text:
        thinking_text = answer_text
        answer_text = f'{answer_text}\n\n**Final Answer:** {ref_ans}'

    raw_full_answer = answer_text
    if not thinking_text and '####' in answer_text:
        steps_part, final_part = answer_text.split('####', 1)
        if len(steps_part.strip()) > 20 and final_part.strip():
            thinking_text = steps_part.strip()
            answer_text = f'#### {final_part.strip()}'
    elif not thinking_text and (
        source_id in ('Roman1111111/gpt-5.4-step-by-step-reasoning', 'OpenDataArena/Spark-234K')
        and len(answer_text) > 200
    ):
        paras = [p.strip() for p in answer_text.split('\n\n') if p.strip()]
        if len(paras) >= 2:
            thinking_text = '\n\n'.join(paras[:-1])
            answer_text = paras[-1]

    if not user_text or not answer_text:
        return None
    return {
        'system': system_text,
        'history': history_turns,
        'question': user_text,
        'thinking': thinking_text,
        'answer': answer_text,
        'raw_answer': raw_full_answer,
        'source': source_id,
        'family': family,
    }

def stream_reasoning_corpus(specs, target_train, target_eval):
    collected = []
    per_source_counts = {}
    for spec in specs:
        ds_id = spec['id']
        fam = spec.get('family', 'reasoning')
        limit = int(spec.get('max_rows', 1500))
        kwargs = {'split': spec.get('split', 'train'), 'streaming': True, 'token': HF_TOKEN}
        if 'config' in spec:
            kwargs['name'] = spec['config']
        if 'data_files' in spec:
            kwargs['data_files'] = spec['data_files']
        count = 0
        cot_count = 0
        try:
            ds = load_dataset(ds_id, **kwargs)
            for row in ds:
                norm = normalize_reasoning_row(row, ds_id, fam)
                if norm is not None:
                    collected.append(norm)
                    count += 1
                    if norm['thinking']:
                        cot_count += 1
                if count >= limit:
                    break
            key_name = f"{ds_id}:{spec.get('config', spec.get('split', 'train'))}"
            per_source_counts[key_name] = {'rows': count, 'with_cot': cot_count, 'family': fam}
            print(f'  Loaded {count:5d} rows (CoT={cot_count:5d}) from [{fam}] {key_name}')
        except Exception as exc:
            print(f'  WARNING: Skipped {ds_id} due to {type(exc).__name__}: {str(exc)[:140]}')
    if not collected:
        raise RuntimeError('No datasets could be streamed. Check network / Hugging Face connectivity.')
    rng = random.Random(2026)
    rng.shuffle(collected)
    eval_slice = collected[:target_eval]
    train_slice = collected[target_eval:target_eval + target_train]
    return train_slice, eval_slice, per_source_counts

print('Streaming curated Claude Fable/Opus + GPT Astra/5/6 + Science Reasoning + Multi-Turn Chat mixture...')
train_rows, eval_rows, DATASET_STATS = stream_reasoning_corpus(
    CFG['REASONING_DATASETS'], CFG['TRAIN_EXAMPLES'], CFG['EVAL_EXAMPLES']
)
print(f'Total collected: {len(train_rows)} train rows, {len(eval_rows)} held-out eval rows across {len(DATASET_STATS)} sources.')


In [ ]:
# Supervised Instruction / CoT Finetuning Assistance via OpenCode (`OPENCODE_API_KEY`)
# Rotates across `deepseek-v4-flash`, `haiku5.5`, and `mimo-v2.6-pro`:
#   1) Verifies GSM8K numeric final answers (`#### NUMBER`) while upgrading to rich `<think>...</think>` traces.
#   2) Enriches non-thinking chat rows with faithful `<think>...</think>` reasoning traces.
def final_number(text):
    if '####' in text: text=text.split('####')[-1]
    nums=re.findall(r'-?\d+(?:,\d{3})*(?:\.\d+)?',text)
    if not nums: return None
    try: return str(Decimal(nums[-1].replace(',','')).normalize())
    except InvalidOperation: return None

if CFG['ENABLE_TEACHER']:
    from openai import OpenAI
    client = OpenAI(api_key=OPENCODE_API_KEY, base_url=CFG['OPENCODE_BASE_URL'], timeout=75.0, max_retries=1)
    teacher_models = list(CFG.get('TEACHER_MODELS') or ['deepseek-v4-flash', 'haiku5.5', 'mimo-v2.6-pro'])
    verified_math, enriched_chat, rejected = 0, 0, 0

    def call_opencode_teacher(messages, idx_hint=0):
        for offset in range(len(teacher_models)):
            m_name = teacher_models[(idx_hint + offset) % len(teacher_models)]
            for candidate in (m_name, f'anthropic/{m_name}' if 'haiku' in m_name else m_name):
                try:
                    resp = client.chat.completions.create(
                        model=candidate,
                        max_tokens=CFG['TEACHER_MAX_TOKENS'],
                        messages=messages,
                    )
                    txt = (resp.choices[0].message.content or '').strip()
                    if txt:
                        return txt, candidate
                except Exception:
                    continue
        return None, None

    # Part A: Verified math CoT synthesis on GSM8K rows
    gsm_candidates = [ex for ex in train_rows if 'gsm8k' in ex.get('source', '').lower()]
    half_budget = max(1, CFG['TEACHER_EXAMPLES'] // 2)
    for i, ex in enumerate(gsm_candidates[:half_budget]):
        suggestion, used_model = call_opencode_teacher([
            {'role': 'system', 'content': 'Solve the math problem step by step. Put your full reasoning inside <think>...</think> and end your final answer with #### NUMBER.'},
            {'role': 'user', 'content': ex['question']},
        ], idx_hint=i)
        if not suggestion:
            rejected += 1
            continue
        gold = final_number(ex.get('raw_answer') or ex['answer'])
        got = final_number(suggestion)
        if gold is not None and got == gold:
            t_in, a_in = split_inline_thinking(suggestion)
            if t_in and a_in:
                ex['thinking'] = t_in
                ex['answer'] = a_in
            else:
                ex['thinking'] = suggestion
            ex['source'] = f'opencode-{used_model}-verified'
            verified_math += 1
        else:
            rejected += 1

    # Part B: CoT reasoning enrichment for chat rows lacking <think>
    non_cot_candidates = [ex for ex in train_rows if not ex.get('thinking')]
    for i, ex in enumerate(non_cot_candidates[:CFG['TEACHER_EXAMPLES'] - half_budget]):
        suggestion, used_model = call_opencode_teacher([
            {'role': 'system', 'content': 'Given the user prompt and reference response, write a concise internal reasoning chain inside <think>...</think> that plans and verifies the response. Output ONLY the <think>...</think> block.'},
            {'role': 'user', 'content': f"User prompt:\n{ex['question']}\n\nReference response:\n{ex['answer'][:1500]}"},
        ], idx_hint=i)
        if suggestion:
            t_in, _ = split_inline_thinking(suggestion)
            cot_body = t_in if t_in else suggestion.replace('<think>', '').replace('</think>', '').strip()
            if len(cot_body) >= 32:
                ex['thinking'] = cot_body
                ex['source'] = f"{ex['source']}+opencode-{used_model}"
                enriched_chat += 1
        else:
            rejected += 1
    print(f'OpenCode SFT Assistant ({teacher_models}): {verified_math} math verified, {enriched_chat} chat CoT enriched, {rejected} skipped')

In [ ]:
# Native Spark-X2.6 <think>...</think> tokenization in 130 GB System RAM + Sequence Packing.
CLOSE_THINK_IDS = tok('</think>', add_special_tokens=False)['input_ids']
if CLOSE_THINK_IDS and isinstance(CLOSE_THINK_IDS[0], list):
    CLOSE_THINK_IDS = CLOSE_THINK_IDS[0]

def tokenized_example(ex, max_length=CFG['MAX_SEQ_LENGTH']):
    has_cot = bool(ex.get('thinking'))
    msgs = []
    if ex.get('system'):
        msgs.append({'role': 'system', 'content': ex['system']})
    for h_msg in ex.get('history', []):
        msgs.append({'role': h_msg['role'], 'content': h_msg['content']})
    msgs.append({'role': 'user', 'content': ex['question']})
    prompt_ids = tok.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        enable_thinking=has_cot, reasoning_effort='high')
    if hasattr(prompt_ids, 'input_ids'): prompt_ids = prompt_ids.input_ids
    elif isinstance(prompt_ids, dict) and 'input_ids' in prompt_ids: prompt_ids = prompt_ids['input_ids']
    if isinstance(prompt_ids, torch.Tensor): prompt_ids = prompt_ids.flatten().tolist()
    elif prompt_ids and isinstance(prompt_ids[0], list): prompt_ids = prompt_ids[0]
    avail = max_length - len(prompt_ids)
    if avail < 64:
        return None
    answer_ids = tok(ex['answer'], add_special_tokens=False)['input_ids']
    if answer_ids and isinstance(answer_ids[0], list): answer_ids = answer_ids[0]
    if len(answer_ids) < 2:
        return None
    if has_cot:
        think_ids = tok(ex['thinking'], add_special_tokens=False)['input_ids']
        if think_ids and isinstance(think_ids[0], list): think_ids = think_ids[0]
        reserve_ans = min(len(answer_ids) + 1 + len(CLOSE_THINK_IDS), max(32, avail // 3))
        max_think = max(16, avail - reserve_ans)
        think_ids = think_ids[:max_think]
        target_ids = think_ids + CLOSE_THINK_IDS + answer_ids + [tok.eos_token_id]
    else:
        target_ids = answer_ids + [tok.eos_token_id]
    full_target_ids = target_ids[:avail]
    if len(full_target_ids) < 2:
        return None
    ids = prompt_ids + full_target_ids
    labels = [-100]*len(prompt_ids) + full_target_ids
    return {'input_ids':ids,'labels':labels,'source':ex['source'],'family':ex.get('family','reasoning'),'has_cot':has_cot}

def pack_tokenized_sequences(items, max_length=CFG['MAX_SEQ_LENGTH']):
    """Pack shorter tokenized CoT conversations into full max_length blocks in System RAM
    while masking boundaries (`labels=-100` on prompt tokens and across EOS transitions).
    """
    packed = []
    cur_ids, cur_labels = [], []
    cur_cot = False
    for it in items:
        ids, labs = it['input_ids'], it['labels']
        if len(cur_ids) + len(ids) <= max_length:
            # Mask the first token of a concatenated document so the previous EOS does not predict the next BOS
            masked_labs = [-100] + labs[1:] if cur_ids else labs
            cur_ids.extend(ids)
            cur_labels.extend(masked_labs)
            cur_cot = cur_cot or it.get('has_cot', False)
        else:
            if cur_ids:
                packed.append({'input_ids': cur_ids, 'labels': cur_labels, 'source': 'packed', 'has_cot': cur_cot})
            cur_ids = list(ids)
            cur_labels = list(labs)
            cur_cot = bool(it.get('has_cot', False))
    if cur_ids:
        packed.append({'input_ids': cur_ids, 'labels': cur_labels, 'source': 'packed', 'has_cot': cur_cot})
    return packed

num_workers = min(8, max(2, (os.cpu_count() or 4)))
with ThreadPoolExecutor(max_workers=num_workers) as pool:
    raw_train_data = [x for x in pool.map(tokenized_example, train_rows) if x is not None]
    eval_data = [x for x in pool.map(tokenized_example, eval_rows) if x is not None]

assert raw_train_data and eval_data, 'No valid tokenized examples after length filtering.'
cot_ratio = sum(1 for x in raw_train_data if x['has_cot']) / max(1, len(raw_train_data))
train_data = pack_tokenized_sequences(raw_train_data, CFG['MAX_SEQ_LENGTH']) if CFG.get('PACK_SEQUENCES', True) else raw_train_data
print(f'Tokenized in System RAM: {len(raw_train_data)} raw -> {len(train_data)} packed train sequences, {len(eval_data)} eval | CoT ratio: {cot_ratio:.1%} | sample lengths: {[len(v["input_ids"]) for v in train_data[:6]]}')

def to_batch(ex):
    return (torch.tensor([ex['input_ids']],dtype=torch.long,device='cuda:0'),
            torch.tensor([ex['labels']],dtype=torch.long,device='cuda:0'))

In [ ]:
# HF export initialization happens before expensive training: fail fast if permissions are inadequate.
from safetensors.torch import save_file
from huggingface_hub import HfApi
import shutil

api=HfApi(token=HF_TOKEN) if HF_TOKEN else None
REPO_ID=CFG['HF_REPO_ID']
if CFG['PUSH_TO_HF']:
    if not REPO_ID:
        owner=api.whoami(token=HF_TOKEN)['name']
        REPO_ID=f'{owner}/Spark-X2.6-4B-Loop-mHC'
    api.create_repo(repo_id=REPO_ID,repo_type='model',private=CFG['HF_PRIVATE'],exist_ok=True,token=HF_TOKEN)
    info=api.repo_info(repo_id=REPO_ID,repo_type='model',token=HF_TOKEN)
    if CFG['HF_PRIVATE'] and not info.private:
        raise RuntimeError('Output repo already exists as PUBLIC; choose another name or change privacy before training.')
    print('Export repo initialized:',REPO_ID,'(private:',info.private,')')
else:
    print('WARNING: no Hub push; local session files may disappear. Turn PUSH_TO_HF on.')

OUT_DIR=pathlib.Path('/content/sparkx_loop_exports')
OUT_DIR.mkdir(parents=True,exist_ok=True)

def export_stage(stage_name, global_steps, metrics=None):
    folder=OUT_DIR/stage_name
    folder.mkdir(parents=True,exist_ok=True)
    state=trainable_state(net)
    save_file(state, str(folder/'trainable.safetensors'))
    metadata={
        'family':'SparkX-Loop-v0.2', 'base_id':CFG['MODEL_ID'],
        'base_revision':CFG['MODEL_REVISION'], 'train_mode':CFG['TRAIN_MODE'],
        'num_unique_layers':36,'loop_start':net.loop_start,'loop_end':net.loop_end,
        'loop_passes':net.loop_passes,'total_virtual_layers':net.total_virtual_layers,
        'mhc_enabled':net.mhc_enabled,'dynamic_mhc':CFG['DYNAMIC_MHC'],
        'sublayer_mhc':CFG['SUBLAYER_MHC'],'pass_adapter_rank':CFG['PASS_ADAPTER_RANK'],
        'engram_enabled':net.engram_enabled,'total_optimizer_steps':global_steps,
        'full_trainable_layers':CFG['FULL_LAYERS'],
        'lora_rank':CFG['LORA_RANK'],'lora_alpha':CFG['LORA_ALPHA'],
        'engram_buckets':CFG['ENG_BUCKETS'],'engram_dim':CFG['ENG_DIM'],
        'engram_experts':CFG['ENG_EXPERTS'],'engram_top_k':CFG['ENG_TOP_K'],
        'engram_expert_rank':CFG['ENG_EXPERT_RANK'],'engram_shared_rank':CFG['ENG_SHARED_RANK'],
        'backbone_engram_stride':CFG['BACKBONE_ENGRAM_STRIDE'],
        'max_seq_length':CFG['MAX_SEQ_LENGTH'],
        'metrics':metrics or {}, 'dataset_stats':DATASET_STATS,
        'teacher_enabled':CFG['ENABLE_TEACHER'],
        'teacher_models':CFG.get('TEACHER_MODELS', []),
        'notes':'Full/hybrid SparkX-Loop v0.2 export with Sublayer Dual-mHC, Pass Adapters, Multi-Scale Engram, and 1M-Context SDPA.'
    }
    (folder/'loop_config.json').write_text(json.dumps(metadata,indent=2))
    shutil.copyfile('sparkx_loop_core.py',folder/'sparkx_loop_core.py')
    shutil.copyfile('load_inference.py',folder/'load_inference.py')
    (folder/'README.md').write_text(
      f'# SparkX-Loop {stage_name} ({REPO_ID})\n\nLoad base `{CFG["MODEL_ID"]}`; attach matching LoRA; '
      'instantiate SparkXLoop, then load `trainable.safetensors` with strict=False. '
      'Use `generate_cached` (via `_VisitCache`) or `generate_slow`; preserves Spark-X2.6-4B 1M context window.\n'
    )
    if CFG['PUSH_TO_HF']:
        api.upload_folder(repo_id=REPO_ID,repo_type='model',
                          folder_path=str(folder),path_in_repo=f'stages/{stage_name}',
                          commit_message=f'SparkX-Loop export: {stage_name} (step {global_steps})', token=HF_TOKEN)
    print('EXPORTED:', stage_name, 'tensors=',len(state), 'repo=',REPO_ID if CFG['PUSH_TO_HF'] else str(folder))
    del state
    return str(folder)

In [ ]:
# Unsloth / bitsandbytes PagedAdamW8bit (pages to 130 GB System RAM if needed) + Cosine LR Schedule
opt = build_optimizer(
    net,
    lr_adapter=CFG['LR_ADAPTER'],
    lr_full=CFG['LR_FULL'],
    lr_mhc=CFG['LR_MHC'],
    use_8bit=CFG.get('USE_8BIT_OPTIM', True),
)
total_planned_steps = sum(s['steps'] for s in CFG['STAGES'])
warmup_steps = max(20, int(0.05 * total_planned_steps))
def lr_lambda(current_step: int):
    if current_step < warmup_steps:
        return float(current_step + 1) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_planned_steps - warmup_steps))
    return max(0.1, 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress))))
scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
print('Optimizer initialized:', type(opt).__name__, '| Total planned steps:', total_planned_steps, '| Warmup:', warmup_steps)

def evaluate(max_examples):
    net.eval()
    total_loss=0.0; n=0
    for ex in eval_data[:max_examples]:
        ids,lab=to_batch(ex)
        with torch.no_grad():
            value=net(ids,labels=lab)['loss']
        total_loss += float(value.item());n+=1
    return total_loss/max(1,n)

net.set_loop(4,12,1,use_mhc=False,use_engram=False)
initial_base_loss=evaluate(CFG['EVAL_LIMIT_PER_STAGE'])
print('Baseline (original path) loss on eval subset:',round(initial_base_loss,4))

In [ ]:
# === RUN ALL STAGES: progressive curriculum with periodic checkpointing for 23-hour Colab G4 session ===
metrics_log=[]
global_step=0
run_interrupted=False
last_export=None
save_every = int(CFG.get('SAVE_EVERY_STEPS', 250))
try:
    for stage in CFG['STAGES']:
        if time.monotonic()-STARTED>CFG['MAX_RUN_HOURS']*3600:
            print('Wallclock budget reached before stage',stage['name']);break
        net.set_loop(stage['start'],stage['end'],stage['passes'],use_mhc=True,use_engram=CFG['ENGRAM'])
        net.train()
        print('START:',stage['name'],'| distinct=',36,'effective=',net.total_virtual_layers)
        stage_losses=[]
        for step in range(stage['steps']):
            if time.monotonic()-STARTED>CFG['MAX_RUN_HOURS']*3600:
                run_interrupted=True;print('Wallclock cap reached; exporting current stage.');break
            opt.zero_grad(set_to_none=True)
            accum_loss=0.0
            for micro in range(CFG['GRAD_ACCUM_STEPS']):
                ex=random.choice(train_data)
                input_ids, labels=to_batch(ex)
                loss=net(input_ids, labels=labels)['loss']/CFG['GRAD_ACCUM_STEPS']
                if not torch.isfinite(loss):
                    raise FloatingPointError(f'Nonfinite training loss at stage={stage["name"]} step={step}')
                loss.backward()
                accum_loss+=float(loss.item())
            torch.nn.utils.clip_grad_norm_((p for p in net.parameters() if p.requires_grad),1.0)
            opt.step(); scheduler.step(); global_step+=1
            stage_losses.append(accum_loss)
            if step==0 or (step+1)%10==0:
                attn_gates=[torch.tanh(net.controllers[str(i)].extra_gain.detach()).abs().item()
                            for i in range(stage['start'],stage['end'])]
                ffn_gates=[torch.tanh(net.controllers[str(i)].ffn_extra_gain.detach()).abs().item()
                           for i in range(stage['start'],stage['end'])]
                mean_ag=sum(attn_gates)/max(1,len(attn_gates))
                mean_fg=sum(ffn_gates)/max(1,len(ffn_gates))
                elapsed_h=(time.monotonic()-STARTED)/3600
                print(f'  {stage["name"]} step {step+1}/{stage["steps"]} (global {global_step}, {elapsed_h:.2f}h): '
                      f'train_loss={accum_loss:.4f}; |attn_gate|={mean_ag:.4f}; |ffn_gate|={mean_fg:.4f}; '
                      f'peak_vram={torch.cuda.max_memory_allocated()/2**30:.1f} GiB')
            if save_every > 0 and (step + 1) % save_every == 0 and (step + 1) < stage['steps']:
                last_export = export_stage(stage['name'], global_step, {
                    'stage': stage['name'],
                    'partial_step': step + 1,
                    'recent_train_loss': sum(stage_losses[-save_every:]) / min(len(stage_losses), save_every),
                    'elapsed_h': (time.monotonic() - STARTED) / 3600,
                })
                net.train()
        stage_eval=evaluate(CFG['EVAL_LIMIT_PER_STAGE'])
        attn_gates=[torch.tanh(net.controllers[str(i)].extra_gain.detach()).abs().item()
                    for i in range(stage['start'],stage['end'])]
        ffn_gates=[torch.tanh(net.controllers[str(i)].ffn_extra_gain.detach()).abs().item()
                   for i in range(stage['start'],stage['end'])]
        record={
            'stage':stage['name'],
            'mean_train_loss':sum(stage_losses)/max(len(stage_losses),1),
            'eval_loss':stage_eval,
            'mean_abs_attn_gate':sum(attn_gates)/max(1,len(attn_gates)),
            'mean_abs_ffn_gate':sum(ffn_gates)/max(1,len(ffn_gates)),
            'engram_gate':torch.sigmoid(net.engram.gain.detach()).item() if CFG['ENGRAM'] else 0.0,
            'optimizer_steps':global_step,
            'elapsed_h':(time.monotonic()-STARTED)/3600,
        }
        metrics_log.append(record)
        print('METRICS:',record)
        if CFG['UPLOAD_EACH_STAGE'] or run_interrupted or stage==CFG['STAGES'][-1]:
            last_export=export_stage(stage['name'],global_step,record)
        gc.collect();torch.cuda.empty_cache()
        if run_interrupted:break
except Exception as err:
    print('Training stopped by',type(err).__name__,str(err)[:300])
    try: last_export=export_stage('partial_on_exception',global_step,{'error_type':type(err).__name__})
    except Exception as ex: print('Partial export also failed:',type(ex).__name__)
    raise
finally:
    print('Completed optimizer steps:',global_step,'elapsed hours:',round((time.monotonic()-STARTED)/3600,2))

In [ ]:
# Final selection, sample generation (cached via _VisitCache + uncached verification), and export metrics summary.
print('Summary:')
for item in metrics_log: print(item)
metrics_path=OUT_DIR/'one_shot_metrics.json'
metrics_path.write_text(json.dumps({'config':CFG,'baseline_eval_loss':initial_base_loss,
    'dataset_stats':DATASET_STATS,'stages':metrics_log,'final_artifact':last_export},indent=2))
if CFG['PUSH_TO_HF']:
    api.upload_file(repo_id=REPO_ID,repo_type='model',path_or_fileobj=str(metrics_path),
                    path_in_repo='one_shot_metrics.json',token=HF_TOKEN)
    print('All metrics uploaded to:',f'https://huggingface.co/{REPO_ID}')
else: print('Local output folder:',OUT_DIR)

net.eval()
try:
    prompt_ids=tok.apply_chat_template(
        [{'role':'user','content':'A store sells notebooks for $4 each and pens for $2 each. If Maria buys 8 items for $26 total, how many notebooks did she buy?'}],
        tokenize=True,add_generation_prompt=True,enable_thinking=True,reasoning_effort='high',return_tensors='pt')
    if hasattr(prompt_ids, 'input_ids'): prompt_ids = prompt_ids.input_ids
    elif isinstance(prompt_ids, dict): prompt_ids = prompt_ids['input_ids']
    prompt_ids = prompt_ids.to('cuda:0')
    cached_ids=net.generate_cached(prompt_ids,max_new_tokens=192,eos_token_id=tok.eos_token_id)
    print('Cached looped CoT sample (<think>...):',tok.decode(cached_ids[0, prompt_ids.shape[-1]:],skip_special_tokens=False))
except Exception as ex:
    print('Optional sample generation skipped:',type(ex).__name__,str(ex)[:140])
print('ONE-SHOT RUN COMPLETE. Colab runtime storage is disposable; exported weights/metrics are on HF if enabled.')